# MASOR-2.1 — Clean annotation-aware ordinal histology segmentation

This notebook is a clean rebuild of MASOR-2 designed to fix the implementation issues found in the previous run **without adding unrelated architecture tricks**.

### What changes versus MASOR-2 v1

1. **Binary localisation and grading are decoupled.** The genuine EBHI binary mask supervises localisation; grading is learned only inside foreground.
2. **Ordinal prototypes are truly trainable model parameters.** They are part of `model.parameters()` from construction time and are gradient-audited on the first batch.
3. **Stain InfoNCE is feature-level, not logit-level.** Contrastive learning acts on the shared morphology feature map.
4. **SoftGrade uses an EMA teacher.** Weak pixel labels are refined only after warm-up, with ordinal-adjacent pseudo-targets.
5. **Severe-region pooling is actually implemented.** Image-level grade supervision uses foreground-masked log-sum-exp pooling.
6. **HGD receives a neighbour-margin loss** against LGD and adenocarcinoma rather than a crude manual class multiplier.
7. **No HGD copy-paste.** Imbalance is handled with a group-aware weighted sampler and mild fold-specific class weighting.
8. **Checkpoint selection is class-balanced.** `best.pt` is selected by macro foreground Dice + geometric-mean foreground Dice.
9. **Inference uses validation-only calibration** of binary threshold and image-prior strength; the outer test fold is untouched.
10. **Stain robustness is evaluated without stain normalisation** so the learned invariance is not confounded by preprocessing.

> Important annotation note: EBHI supplies a binary foreground mask and an image-level grade. Dense grade maps are therefore **weakly supervised latent maps**. For benchmark comparability, the evaluation semantic target is the binary mask painted with the image-level grade; this is not claimed to be independently pathologist-graded at every pixel.


In [ ]:
# 0. Install + imports + deterministic setup
import sys, subprocess, os, re, json, math, random, hashlib, platform, time, copy
from pathlib import Path
from collections import defaultdict, Counter

PINNED = {
    "timm": "1.0.19",
    "scikit-learn": "1.5.2",
    "scikit-image": "0.24.0",
}
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
    *(f"{k}=={v}" for k, v in PINNED.items())
])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import sklearn
from sklearn.model_selection import StratifiedGroupKFold
from skimage.color import rgb2hed, hed2rgb
from IPython.display import display

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = (DEVICE.type == "cuda")
PIN_MEMORY = (DEVICE.type == "cuda")
print("Python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__, "|", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available. Kaggle: Settings > Accelerator > GPU for the full 5-fold run.")


### v2 runtime patch
This revision avoids pandas `Series.mask` name collisions by using bracket column access (`row["mask"]`), uses the current `torch.amp` API, and only enables pinned memory when CUDA is available.


In [ ]:
# 1. Configuration
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5
FOLDS_TO_RUN = [0, 1, 2, 3, 4]

IMAGE_SIZE = 224
SOURCE_MASK_THRESHOLD = 128

# Semantic evaluation order is deliberately ordinal after background.
SEM_CLASS_NAMES = ["background", "normal_gland", "low_grade", "high_grade", "adk_invasion"]
GRADE_NAMES = ["normal_gland", "low_grade", "high_grade", "adk_invasion"]
GRADE_ORDER = {n: i for i, n in enumerate(GRADE_NAMES)}
SEM_INDEX = {n: i for i, n in enumerate(SEM_CLASS_NAMES)}
N_GRADES = len(GRADE_NAMES)

FOLDER_TO_CLASS = {
    "Adenocarcinoma": "adk_invasion",
    "High-grade IN": "high_grade",
    "Low-grade IN": "low_grade",
    "Normal": "normal_gland",
}
KNOWN_ORPHANS = {
    ("low_grade", "GT2012149-2-400-001.png"),
    ("low_grade", "GTGT2012149-2-400-001.png"),
}

# ---- Backbone / model ----
BACKBONE = "convnext_tiny"
BACKBONE_PRETRAINED = True
FPN_DIM = 128
PROJ_DIM = 64
PROTO_TEMP = 0.20
PROTO_RESIDUAL_SCALE = 0.05
PROTO_MIN_GAP = 0.10

# ---- Training ----
EPOCHS = 70
PATIENCE = 14
BATCH_SIZE = 8
NUM_WORKERS = 2
LR = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 2.0

# Group-aware sampling. alpha=0 -> natural, alpha=1 -> aggressive inverse-frequency.
SAMPLER_CLASS_ALPHA = 0.50
SAMPLER_GROUP_ALPHA = 0.50
GRADE_CE_ALPHA = 0.25  # mild fold-specific image-grade class balancing

# ---- Loss weights ----
W_BIN_DICE = 0.55
W_BIN_BCE = 0.35
W_BIN_EDGE = 0.10
W_IMAGE_GRADE = 0.80
W_WEAK_PIXEL_WARM = 0.25
W_WEAK_PIXEL_LATE = 0.05
W_SOFTGRADE = 0.30
W_INFO_NCE = 0.10
W_HGD_MARGIN = 0.15
W_PROTO_RESIDUAL = 0.01

# EMA SoftGrade
SOFTGRADE_WARMUP_EPOCHS = 10
EMA_DECAY = 0.995
SOFTGRADE_CONF_POWER = 2.0
SOFTGRADE_MIN_WEIGHT = 0.10

# Severe-region pooling
LSE_TAU = 0.20

# HGD-specific ordinal neighbour margin
HGD_MARGIN = 0.20
HGD_INDEX = GRADE_ORDER["high_grade"]
LGD_INDEX = GRADE_ORDER["low_grade"]
ADC_INDEX = GRADE_ORDER["adk_invasion"]

# Feature-level stain InfoNCE
INFO_NCE_WARMUP_EPOCHS = 3
INFO_NCE_TEMPERATURE = 0.10
INFO_NCE_MAX_ANCHORS = 256

# H&E perturbation for the paired stain view (no stain normalisation in the full model)
STAIN_H_SCALE = (0.80, 1.20)
STAIN_E_SCALE = (0.80, 1.20)
STAIN_H_SHIFT = (-0.025, 0.025)
STAIN_E_SHIFT = (-0.025, 0.025)
STAIN_EVAL_REPEATS = 3

# Validation-only calibration grid
BIN_THRESHOLDS = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
PRIOR_LAMBDAS = [0.00, 0.25, 0.50, 0.75]
DEFAULT_BIN_THRESHOLD = 0.50
DEFAULT_PRIOR_LAMBDA = 0.50

# TTA for the final val/test pass only. Training-time checkpoint selection uses no TTA.
TTA_ENABLE = True
TTA_OPS = ["id", "hflip", "vflip", "rot90"]

BOOTSTRAP_ITERS = 5000
EXPERIMENT_TAG = "masor21_clean"
WORK_ROOT = Path(f"/kaggle/working/ebhi_{EXPERIMENT_TAG}")
ARTIFACT_ROOT = WORK_ROOT / "artifacts"
CHECKPOINT_ROOT = WORK_ROOT / "checkpoints"
for p in [WORK_ROOT, ARTIFACT_ROOT, CHECKPOINT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "backbone": BACKBONE,
    "pretrained": BACKBONE_PRETRAINED,
    "binary_grade_decoupled": True,
    "ordered_prototypes": True,
    "feature_stain_infonce": True,
    "ema_softgrade": True,
    "hgd_margin": True,
    "copy_paste": False,
    "stain_normalisation": False,
    "tta": TTA_ENABLE,
}, indent=2))


In [ ]:
# 2. Locate EBHI-SEG robustly
import subprocess as _sp

def _find_ebhi_root():
    candidates = [
        "/kaggle/input/ebhi-seg-colorectal-cancer/EBHI-SEG",
        "/kaggle/input/datasets/orvile/ebhi-seg-colorectal-cancer/EBHI-SEG",
        "/kaggle/input/datasets/sarasova/ebhi-seg-colorectal-cancer/EBHI-SEG",
        "./EBHI-SEG",
    ]
    for cand in candidates:
        p = Path(cand)
        if p.is_dir() and (p / "Adenocarcinoma").is_dir():
            return p

    # Last-resort search, but still require the four expected class folders under one parent.
    for base in ("/kaggle", "/root", "/home", "/tmp", "/opt"):
        if not Path(base).is_dir():
            continue
        try:
            out = _sp.run(
                ["find", base, "-type", "d", "-name", "Adenocarcinoma"],
                capture_output=True, text=True, timeout=90
            ).stdout
            for line in out.splitlines():
                line = line.strip()
                if not line:
                    continue
                parent = Path(line).parent
                names = {c.name.lower() for c in parent.iterdir() if c.is_dir()}
                if {n.lower() for n in FOLDER_TO_CLASS}.issubset(names):
                    return parent
        except Exception:
            pass
    return None

ROOT = _find_ebhi_root()
if ROOT is None:
    raise FileNotFoundError("EBHI-SEG not mounted. Attach the dataset in Kaggle and restart the session.")
print("Dataset ROOT:", ROOT)


In [ ]:
# 3. Deterministic master manifest + group/duplicate audit

def find_image_and_mask_dirs(class_dir):
    image_names = {"image", "images", "img"}
    mask_names = {"label", "labels", "mask", "masks", "gt", "groundtruth", "ground_truth", "ground truth"}
    subdirs = [p for p in class_dir.iterdir() if p.is_dir()]
    img_dir = next((p for p in subdirs if p.name.lower() in image_names), None)
    mask_dir = next((p for p in subdirs if p.name.lower() in mask_names), None)
    if img_dir is None or mask_dir is None:
        raise FileNotFoundError(f"Missing image/mask directories under {class_dir}: {[p.name for p in subdirs]}")
    return img_dir, mask_dir


def derive_group_key(stem):
    s = stem.strip()
    pos = s.upper().find("GT")
    if pos >= 0:
        s = s[pos:]
    s = re.sub(r"[ _]+", "-", s)
    m = re.match(r"^(GT[A-Z]*\d+)-(\d+)-(100|200|400)-(\d+)$", s, flags=re.I)
    if m:
        return f"{m.group(1)}-{m.group(2)}"
    key = re.sub(r"-(100|200|400)-\d+$", "", s, flags=re.I)
    if key != s:
        return key
    key = re.sub(r"[-_]\d+$", "", s)
    return key or s


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()

records, exclusions = [], []
for folder_name, cls in FOLDER_TO_CLASS.items():
    class_dir = next((p for p in ROOT.iterdir() if p.is_dir() and p.name.lower() == folder_name.lower()), None)
    if class_dir is None:
        raise FileNotFoundError(f"Missing class folder: {folder_name}")
    img_dir, mask_dir = find_image_and_mask_dirs(class_dir)
    masks = {p.name: p for p in sorted(mask_dir.glob("*.png"))}
    for img_path in sorted(img_dir.glob("*.png")):
        mask_path = masks.get(img_path.name)
        if mask_path is None:
            exclusions.append({"target_class": cls, "image_name": img_path.name, "reason": "missing_exact_same_named_mask"})
            continue
        records.append({
            "sample_id": f"{cls}__{img_path.stem}",
            "stem": img_path.stem,
            "target_class": cls,
            "grade_idx": GRADE_ORDER[cls],
            "source_group_key": derive_group_key(img_path.stem),
            "image": str(img_path),
            "mask": str(mask_path),
            "source_image_sha256": sha256_file(img_path),
            "source_mask_sha256": sha256_file(mask_path),
        })

exclusions = pd.DataFrame(exclusions)
if len(exclusions):
    found = set(zip(exclusions.target_class, exclusions.image_name))
    unexpected = found - KNOWN_ORPHANS
    assert not unexpected, f"Unexpected missing masks: {sorted(unexpected)}"

master = pd.DataFrame(records).sort_values(["target_class", "source_group_key", "stem"]).reset_index(drop=True)
assert master.sample_id.is_unique
assert set(master.target_class) == set(GRADE_NAMES)

# Merge source groups connected by exact duplicate image hashes.
hash_counts = master.source_image_sha256.value_counts()
dup_hashes = set(hash_counts[hash_counts > 1].index)
parent = {g: g for g in sorted(master.source_group_key.unique())}

def _find(g):
    while parent[g] != g:
        parent[g] = parent[parent[g]]
        g = parent[g]
    return g

def _union(a, b):
    ra, rb = _find(a), _find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

for _, g in master[master.source_image_sha256.isin(dup_hashes)].groupby("source_image_sha256"):
    keys = sorted(g.source_group_key.unique())
    for other in keys[1:]:
        _union(keys[0], other)
master["group_key"] = master.source_group_key.map(_find)

MASTER_MANIFEST = WORK_ROOT / "master_manifest.csv"
master.to_csv(MASTER_MANIFEST, index=False)
print(f"Retained {len(master):,} images | {master.group_key.nunique():,} groups")
display(master.groupby("target_class").agg(images=("sample_id", "size"), groups=("group_key", "nunique")))


In [ ]:
# 4. Locked nested grouped splits

def build_outer_assignments(df):
    sgkf = StratifiedGroupKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=SEED)
    outer = np.full(len(df), -1, dtype=int)
    for f, (_, ti) in enumerate(sgkf.split(df, y=df.target_class, groups=df.group_key)):
        outer[ti] = f
    assert (outer >= 0).all()
    return outer

master["outer_fold"] = build_outer_assignments(master)
master.to_csv(MASTER_MANIFEST, index=False)
MASTER_SHA256 = hashlib.sha256(MASTER_MANIFEST.read_bytes()).hexdigest()
print("Master manifest SHA256:", MASTER_SHA256)


def make_fold_manifest(fold):
    mf = master.copy()
    test_mask = mf.outer_fold.eq(fold)
    dev_idx = np.flatnonzero(~test_mask.to_numpy())
    dev = mf.iloc[dev_idx]
    inner = StratifiedGroupKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=SEED + 1000 + fold)
    tr_rel, va_rel = next(inner.split(dev, y=dev.target_class, groups=dev.group_key))
    tr_idx, va_idx = dev_idx[tr_rel], dev_idx[va_rel]
    te_idx = np.flatnonzero(test_mask.to_numpy())
    mf["split"] = ""
    mf.loc[tr_idx, "split"] = "train"
    mf.loc[va_idx, "split"] = "val"
    mf.loc[te_idx, "split"] = "test"
    for s in ["train", "val", "test"]:
        part = mf[mf.split == s]
        assert set(part.target_class) == set(GRADE_NAMES)
    # hard no-leak check
    tr_g = set(mf[mf.split == "train"].group_key)
    va_g = set(mf[mf.split == "val"].group_key)
    te_g = set(mf[mf.split == "test"].group_key)
    assert not (tr_g & va_g or tr_g & te_g or va_g & te_g)
    return mf


## Data policy

The dataset returns **two views of the same geometry**:

- `clean`: the spatially augmented H&E tile;
- `stain`: the exact same pixels after an H&E-channel perturbation.

The binary mask is transformed identically for both. No histogram matching or stain normalisation is applied in this notebook, so the contrastive stain result can be attributed to learned invariance rather than preprocessing.


In [ ]:
# 5. Dataset, paired H&E perturbation, and group-aware sampler
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def resize_image_and_mask(image, mask, size=IMAGE_SIZE):
    img = Image.fromarray(image).resize((size, size), resample=Image.Resampling.BILINEAR)
    msk = Image.fromarray(mask).resize((size, size), resample=Image.Resampling.NEAREST)
    return np.asarray(img), np.asarray(msk)


def shared_spatial_aug(image, mask):
    k = np.random.randint(0, 4)
    if k:
        image = np.rot90(image, k).copy()
        mask = np.rot90(mask, k).copy()
    if np.random.rand() < 0.5:
        image = np.fliplr(image).copy(); mask = np.fliplr(mask).copy()
    if np.random.rand() < 0.5:
        image = np.flipud(image).copy(); mask = np.flipud(mask).copy()
    return image, mask


def hed_stain_perturb(image_uint8):
    x = image_uint8.astype(np.float32) / 255.0
    hed = rgb2hed(np.clip(x, 1e-6, 1.0))
    h_scale = np.random.uniform(*STAIN_H_SCALE)
    e_scale = np.random.uniform(*STAIN_E_SCALE)
    h_shift = np.random.uniform(*STAIN_H_SHIFT)
    e_shift = np.random.uniform(*STAIN_E_SHIFT)
    hed[..., 0] = hed[..., 0] * h_scale + h_shift
    hed[..., 1] = hed[..., 1] * e_scale + e_shift
    out = hed2rgb(hed)
    out = np.clip(out, 0.0, 1.0)
    return (out * 255.0).astype(np.uint8)


def to_tensor_norm(image_uint8):
    x = image_uint8.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(x.transpose(2, 0, 1)).float()


class EBHIWeakOrdinalDataset(Dataset):
    def __init__(self, df, train=False, paired_stain=True):
        self.df = df.reset_index(drop=True).copy()
        self.train = train
        self.paired_stain = paired_stain

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = np.asarray(Image.open(row["image"]).convert("RGB"))
        mask = np.asarray(Image.open(row["mask"]).convert("L"))
        image, mask = resize_image_and_mask(image, mask)
        mask = (mask >= SOURCE_MASK_THRESHOLD).astype(np.uint8)

        if self.train:
            image, mask = shared_spatial_aug(image, mask)

        clean = image
        stain = hed_stain_perturb(image) if self.paired_stain else image.copy()

        return {
            "clean": to_tensor_norm(clean),
            "stain": to_tensor_norm(stain),
            "mask": torch.from_numpy(mask[None].astype(np.float32)),
            "grade": torch.tensor(int(row["grade_idx"]), dtype=torch.long),
            "sample_id": row["sample_id"],
            "group_key": row["group_key"],
        }


def build_sampler(train_df):
    class_counts = train_df.target_class.value_counts().to_dict()
    group_counts = train_df.group_key.value_counts().to_dict()
    weights = []
    for _, r in train_df.iterrows():
        wc = class_counts[r["target_class"]] ** (-SAMPLER_CLASS_ALPHA)
        wg = group_counts[r["group_key"]] ** (-SAMPLER_GROUP_ALPHA)
        weights.append(wc * wg)
    weights = torch.as_tensor(weights, dtype=torch.double)
    weights /= weights.mean()
    return WeightedRandomSampler(weights, num_samples=len(train_df), replacement=True)


def fold_grade_ce_weights(train_df):
    counts = train_df.grade_idx.value_counts().reindex(range(N_GRADES), fill_value=1).sort_index().to_numpy(float)
    w = counts ** (-GRADE_CE_ALPHA)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


## Model

The model has one shared ConvNeXt/FPN morphology representation and two outputs:

- a binary foreground logit;
- a dense 64-D morphology feature map.

The four grade logits are **not** an unrelated classifier. They are distances to four trainable ordinal prototypes. Prototype centres are parameterised by positive cumulative gaps on a shared ordinal axis, so their ordering is structural rather than an after-the-fact regulariser.


In [ ]:
# 6. MASOR-2.1 model: binary head + ordered prototype grade field
class FPNDecoder(nn.Module):
    def __init__(self, channels, out_dim=128):
        super().__init__()
        self.lateral = nn.ModuleList([nn.Conv2d(c, out_dim, 1) for c in channels])
        self.smooth = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(out_dim, out_dim, 3, padding=1, bias=False),
                nn.GroupNorm(8, out_dim),
                nn.GELU(),
            ) for _ in channels
        ])

    def forward(self, feats):
        xs = [lat(f) for lat, f in zip(self.lateral, feats)]
        for i in range(len(xs) - 1, 0, -1):
            xs[i - 1] = xs[i - 1] + F.interpolate(xs[i], size=xs[i - 1].shape[-2:], mode="bilinear", align_corners=False)
        xs = [sm(x) for sm, x in zip(self.smooth, xs)]
        return xs[0]


class MASOR21Net(nn.Module):
    def __init__(self, backbone=BACKBONE, pretrained=True):
        super().__init__()
        try:
            self.encoder = timm.create_model(backbone, pretrained=pretrained, features_only=True, out_indices=(0,1,2,3))
        except Exception as e:
            print("WARNING: pretrained backbone load failed; falling back to random init:", repr(e))
            self.encoder = timm.create_model(backbone, pretrained=False, features_only=True, out_indices=(0,1,2,3))
        channels = self.encoder.feature_info.channels()
        self.decoder = FPNDecoder(channels, FPN_DIM)
        self.binary_head = nn.Conv2d(FPN_DIM, 1, 1)
        self.projector = nn.Sequential(
            nn.Conv2d(FPN_DIM, PROJ_DIM, 1, bias=False),
            nn.GroupNorm(8, PROJ_DIM),
            nn.GELU(),
        )

        # Ordered prototype manifold.
        self.ordinal_axis = nn.Parameter(torch.randn(PROJ_DIM) * 0.02)
        self.gap_raw = nn.Parameter(torch.zeros(N_GRADES - 1))
        self.proto_residual = nn.Parameter(torch.zeros(N_GRADES, PROJ_DIM))
        self.grade_bias = nn.Parameter(torch.zeros(N_GRADES))

    def prototypes(self):
        axis = F.normalize(self.ordinal_axis, dim=0)
        gaps = F.softplus(self.gap_raw) + PROTO_MIN_GAP
        centers = torch.cat([torch.zeros(1, device=gaps.device), torch.cumsum(gaps, dim=0)])
        centers = centers - centers.mean()
        protos = centers[:, None] * axis[None, :] + PROTO_RESIDUAL_SCALE * self.proto_residual
        return protos, centers, gaps

    def forward(self, x):
        feats = self.encoder(x)
        f = self.decoder(feats)
        bin_logits = self.binary_head(f)
        z = F.normalize(self.projector(f), dim=1)
        protos, centers, gaps = self.prototypes()
        # B x K x H x W squared distances.
        d = ((z[:, None] - protos[None, :, :, None, None]) ** 2).sum(dim=2)
        grade_logits = -d / PROTO_TEMP + self.grade_bias.view(1, -1, 1, 1)
        return {
            "bin_logits": bin_logits,
            "grade_logits": grade_logits,
            "features": z,
            "prototypes": protos,
            "centers": centers,
            "gaps": gaps,
        }


def audit_optimizer_contains_ordinal_params(model, optimizer):
    opt_ids = {id(p) for g in optimizer.param_groups for p in g["params"]}
    required = {
        "ordinal_axis": model.ordinal_axis,
        "gap_raw": model.gap_raw,
        "proto_residual": model.proto_residual,
        "grade_bias": model.grade_bias,
    }
    missing = [name for name, p in required.items() if id(p) not in opt_ids]
    assert not missing, f"Ordinal parameters missing from optimizer: {missing}"
    print("Optimizer audit passed: all ordinal parameters are trainable.")


In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

row = master.sample(1, random_state=None).iloc[0]
rgb = np.asarray(Image.open(row["image"]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE)))
mask = np.asarray(Image.open(row["mask"]).convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST))
mask = (mask >= 128).astype(np.uint8)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(rgb); ax[0].set_title(f"{row['target_class']}\n{row['sample_id']}")
ax[1].imshow(rgb); ax[1].imshow(np.ma.masked_where(mask==0, mask), alpha=0.5, cmap="autumn")
ax[1].set_title("Ground truth foreground overlay")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# 7. Loss helpers

def soft_dice_loss_from_logits(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    target = target.float()
    inter = (prob * target).sum(dim=(1,2,3))
    den = prob.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2 * inter + eps) / (den + eps)
    return 1 - dice.mean()


def sobel_edges(x):
    kx = torch.tensor([[-1.,0.,1.],[-2.,0.,2.],[-1.,0.,1.]], device=x.device).view(1,1,3,3)
    ky = torch.tensor([[-1.,-2.,-1.],[0.,0.,0.],[1.,2.,1.]], device=x.device).view(1,1,3,3)
    gx = F.conv2d(x, kx, padding=1)
    gy = F.conv2d(x, ky, padding=1)
    return torch.sqrt(gx.pow(2) + gy.pow(2) + 1e-6)


def binary_edge_loss(bin_logits, mask):
    pred = torch.sigmoid(bin_logits)
    return F.l1_loss(sobel_edges(pred), sobel_edges(mask.float()))


def resize_mask(mask, hw):
    return F.interpolate(mask.float(), size=hw, mode="nearest")


def masked_lse_pool(logits, mask, tau=LSE_TAU):
    """Foreground-masked LSE pooling. logits BxKxHxW, mask Bx1xHxW."""
    B, K, H, W = logits.shape
    m = mask[:, 0] > 0.5
    outs = []
    for b in range(B):
        valid = m[b]
        if valid.any():
            vals = logits[b, :, valid]  # K x N
        else:
            vals = logits[b].reshape(K, -1)
        n = vals.shape[1]
        pooled = tau * (torch.logsumexp(vals / tau, dim=1) - math.log(max(n, 1)))
        outs.append(pooled)
    return torch.stack(outs, dim=0)


def feature_infonce(z1, z2, mask_small, max_anchors=INFO_NCE_MAX_ANCHORS, temperature=INFO_NCE_TEMPERATURE):
    """Same-location cross-view positives on morphology features; all sampled locations act as negatives."""
    valid = (mask_small[:,0] > 0.5)
    idx = valid.nonzero(as_tuple=False)  # N x [b,y,x]
    if len(idx) < 2:
        return z1.new_zeros(())
    if len(idx) > max_anchors:
        perm = torch.randperm(len(idx), device=idx.device)[:max_anchors]
        idx = idx[perm]
    a = z1[idx[:,0], :, idx[:,1], idx[:,2]]
    b = z2[idx[:,0], :, idx[:,1], idx[:,2]]
    a = F.normalize(a, dim=1)
    b = F.normalize(b, dim=1)
    sim = a @ b.T / temperature
    target = torch.arange(sim.shape[0], device=sim.device)
    return 0.5 * (F.cross_entropy(sim, target) + F.cross_entropy(sim.T, target))


def ordinal_adjacent_mask(labels, device):
    # Allowed pseudo-target support: image grade and immediate ordinal neighbour(s).
    B = labels.shape[0]
    out = torch.zeros(B, N_GRADES, device=device)
    for b, y in enumerate(labels.tolist()):
        lo, hi = max(0, y - 1), min(N_GRADES - 1, y + 1)
        out[b, lo:hi+1] = 1.0
    return out


def ema_softgrade_loss(student_logits, teacher_logits, mask_small, image_labels):
    with torch.no_grad():
        tp = torch.softmax(teacher_logits, dim=1)
        allow = ordinal_adjacent_mask(image_labels, tp.device)[:, :, None, None]
        tp = tp * allow
        tp = tp / tp.sum(dim=1, keepdim=True).clamp_min(1e-6)
        conf = tp.max(dim=1).values
        weights = SOFTGRADE_MIN_WEIGHT + (1 - SOFTGRADE_MIN_WEIGHT) * conf.pow(SOFTGRADE_CONF_POWER)
    logp = F.log_softmax(student_logits, dim=1)
    ce_soft = -(tp * logp).sum(dim=1)
    fg = mask_small[:,0]
    return (ce_soft * weights * fg).sum() / fg.sum().clamp_min(1.0)


def weak_pixel_ce(grade_logits, mask_small, image_labels, class_weights=None):
    B, K, H, W = grade_logits.shape
    target = image_labels[:, None, None].expand(B, H, W)
    per = F.cross_entropy(grade_logits, target, weight=class_weights, reduction="none")
    fg = mask_small[:,0]
    return (per * fg).sum() / fg.sum().clamp_min(1.0)


def hgd_neighbour_margin_loss(features, protos, mask_small, image_labels, max_points=512):
    losses = []
    for b in range(features.shape[0]):
        if int(image_labels[b]) != HGD_INDEX:
            continue
        valid = (mask_small[b,0] > 0.5).nonzero(as_tuple=False)
        if len(valid) == 0:
            continue
        if len(valid) > max_points:
            valid = valid[torch.randperm(len(valid), device=valid.device)[:max_points]]
        z = features[b, :, valid[:,0], valid[:,1]].T
        d = ((z[:,None,:] - protos[None,:,:]) ** 2).sum(dim=2)
        dh, dl, da = d[:,HGD_INDEX], d[:,LGD_INDEX], d[:,ADC_INDEX]
        loss = F.relu(HGD_MARGIN + dh - dl) + F.relu(HGD_MARGIN + dh - da)
        losses.append(loss.mean())
    if not losses:
        return features.new_zeros(())
    return torch.stack(losses).mean()


def compute_total_loss(model, out_clean, out_stain, batch, epoch, grade_ce_weights, teacher=None):
    mask_full = batch["mask"].to(DEVICE, non_blocking=True)
    labels = batch["grade"].to(DEVICE, non_blocking=True)
    hw = out_clean["bin_logits"].shape[-2:]
    mask_small = resize_mask(mask_full, hw)

    # True binary localisation supervision.
    bce = F.binary_cross_entropy_with_logits(out_clean["bin_logits"], mask_small)
    dice = soft_dice_loss_from_logits(out_clean["bin_logits"], mask_small)
    edge = binary_edge_loss(out_clean["bin_logits"], mask_small)

    # Genuine image-level grade supervision via severe-region pooling.
    image_logits = masked_lse_pool(out_clean["grade_logits"], mask_small, LSE_TAU)
    image_grade = F.cross_entropy(image_logits, labels, weight=grade_ce_weights)

    weak_w = W_WEAK_PIXEL_WARM if epoch < SOFTGRADE_WARMUP_EPOCHS else W_WEAK_PIXEL_LATE
    weak = weak_pixel_ce(out_clean["grade_logits"], mask_small, labels, class_weights=grade_ce_weights)

    soft = out_clean["grade_logits"].new_zeros(())
    if teacher is not None and epoch >= SOFTGRADE_WARMUP_EPOCHS:
        with torch.no_grad():
            t = teacher(batch["clean"].to(DEVICE, non_blocking=True))
        soft = ema_softgrade_loss(out_clean["grade_logits"], t["grade_logits"], mask_small, labels)

    nce = out_clean["grade_logits"].new_zeros(())
    if epoch >= INFO_NCE_WARMUP_EPOCHS:
        nce = feature_infonce(out_clean["features"], out_stain["features"], mask_small)

    hgd_margin = hgd_neighbour_margin_loss(
        out_clean["features"], out_clean["prototypes"], mask_small, labels
    )

    proto_resid = model.proto_residual.pow(2).mean()

    total = (
        W_BIN_DICE * dice +
        W_BIN_BCE * bce +
        W_BIN_EDGE * edge +
        W_IMAGE_GRADE * image_grade +
        weak_w * weak +
        W_SOFTGRADE * soft +
        W_INFO_NCE * nce +
        W_HGD_MARGIN * hgd_margin +
        W_PROTO_RESIDUAL * proto_resid
    )

    parts = {
        "total": total,
        "bin_dice": dice.detach(), "bin_bce": bce.detach(), "bin_edge": edge.detach(),
        "image_grade": image_grade.detach(), "weak_pixel": weak.detach(),
        "softgrade": soft.detach(), "infonce": nce.detach(),
        "hgd_margin": hgd_margin.detach(), "proto_resid": proto_resid.detach(),
    }
    return total, parts


In [ ]:
# 8. EMA + prediction / TTA / semantic metrics
@torch.no_grad()
def update_ema(teacher, student, decay=EMA_DECAY):
    t_state = teacher.state_dict()
    s_state = student.state_dict()
    for k in t_state.keys():
        if torch.is_floating_point(t_state[k]):
            t_state[k].mul_(decay).add_(s_state[k], alpha=1-decay)
        else:
            t_state[k].copy_(s_state[k])


def _apply_tta(x, op):
    if op == "id": return x
    if op == "hflip": return torch.flip(x, dims=[3])
    if op == "vflip": return torch.flip(x, dims=[2])
    if op == "rot90": return torch.rot90(x, 1, dims=[2,3])
    raise ValueError(op)


def _invert_tta(y, op):
    if op == "id": return y
    if op == "hflip": return torch.flip(y, dims=[3])
    if op == "vflip": return torch.flip(y, dims=[2])
    if op == "rot90": return torch.rot90(y, 3, dims=[2,3])
    raise ValueError(op)


@torch.no_grad()
def infer_logits(model, x, tta=False):
    ops = TTA_OPS if (tta and TTA_ENABLE) else ["id"]
    bins, grades = [], []
    for op in ops:
        out = model(_apply_tta(x, op))
        bins.append(_invert_tta(out["bin_logits"], op))
        grades.append(_invert_tta(out["grade_logits"], op))
    return torch.stack(bins).mean(0), torch.stack(grades).mean(0)


@torch.no_grad()
def collect_logits(model, loader, tta=False):
    model.eval()
    out = []
    for batch in loader:
        x = batch["clean"].to(DEVICE, non_blocking=True)
        b, g = infer_logits(model, x, tta=tta)
        out.append({
            "bin_logits": b.cpu(),
            "grade_logits": g.cpu(),
            "mask": batch["mask"].cpu(),
            "grade": batch["grade"].cpu(),
            "sample_id": list(batch["sample_id"]),
            "group_key": list(batch["group_key"]),
        })
    return out


def semantic_from_logits(bin_logits, grade_logits, out_hw, bin_threshold, prior_lambda):
    # logits are on feature resolution.
    bin_prob_small = torch.sigmoid(bin_logits)
    fg_small = (bin_prob_small >= bin_threshold).float()
    image_logits = masked_lse_pool(grade_logits, fg_small, LSE_TAU)
    image_logp = F.log_softmax(image_logits, dim=1)
    fused = grade_logits + prior_lambda * image_logp[:, :, None, None]

    bin_prob = F.interpolate(bin_prob_small, size=out_hw, mode="bilinear", align_corners=False)
    fused_up = F.interpolate(fused, size=out_hw, mode="bilinear", align_corners=False)
    grade_pred = fused_up.argmax(dim=1) + 1  # semantic foreground classes 1..4
    sem = torch.where(bin_prob[:,0] >= bin_threshold, grade_pred, torch.zeros_like(grade_pred))
    return sem, bin_prob[:,0]


def confusion_counts(gt, pred, n_classes=len(SEM_CLASS_NAMES)):
    gt = gt.reshape(-1).long()
    pred = pred.reshape(-1).long()
    cm = torch.bincount(gt * n_classes + pred, minlength=n_classes*n_classes)
    return cm.reshape(n_classes, n_classes)


def metrics_from_cm(cm):
    cm = np.asarray(cm, dtype=np.float64)
    dices = {}
    ious = {}
    for c, name in enumerate(SEM_CLASS_NAMES):
        tp = cm[c,c]
        fp = cm[:,c].sum() - tp
        fn = cm[c,:].sum() - tp
        dices[name] = (2*tp) / max(2*tp + fp + fn, 1e-12)
        ious[name] = tp / max(tp + fp + fn, 1e-12)
    fg = np.array([dices[n] for n in GRADE_NAMES], dtype=float)
    macro = float(fg.mean())
    gmean = float(np.exp(np.log(np.clip(fg, 1e-8, 1)).mean()))
    score = 0.5 * macro + 0.5 * gmean
    return dices, ious, macro, gmean, score


def score_cached(cached, bin_threshold=0.5, prior_lambda=0.5, return_rows=False):
    cm = torch.zeros((len(SEM_CLASS_NAMES), len(SEM_CLASS_NAMES)), dtype=torch.long)
    bin_tp = bin_fp = bin_fn = 0
    rows = []
    for chunk in cached:
        b = chunk["bin_logits"]
        g = chunk["grade_logits"]
        masks = chunk["mask"][:,0].long()
        labels = chunk["grade"].long()
        sem_pred, bin_prob = semantic_from_logits(b, g, masks.shape[-2:], bin_threshold, prior_lambda)
        gt_sem = masks * (labels[:,None,None] + 1)
        cm += confusion_counts(gt_sem, sem_pred)

        bp = (bin_prob >= bin_threshold).long()
        bin_tp += int(((bp == 1) & (masks == 1)).sum())
        bin_fp += int(((bp == 1) & (masks == 0)).sum())
        bin_fn += int(((bp == 0) & (masks == 1)).sum())

        if return_rows:
            for i in range(len(labels)):
                row = {
                    "sample_id": chunk["sample_id"][i],
                    "group_key": chunk["group_key"][i],
                    "true_grade": GRADE_NAMES[int(labels[i])],
                }
                cmi = confusion_counts(gt_sem[i], sem_pred[i])
                for c, name in enumerate(SEM_CLASS_NAMES):
                    tp = int(cmi[c,c]); fp = int(cmi[:,c].sum()-tp); fn = int(cmi[c,:].sum()-tp)
                    row[f"tp_{name}"] = tp; row[f"fp_{name}"] = fp; row[f"fn_{name}"] = fn
                bpi, mi = bp[i], masks[i]
                row["bin_tp"] = int(((bpi==1)&(mi==1)).sum())
                row["bin_fp"] = int(((bpi==1)&(mi==0)).sum())
                row["bin_fn"] = int(((bpi==0)&(mi==1)).sum())
                rows.append(row)

    dices, ious, macro, gmean, score = metrics_from_cm(cm.numpy())
    binary_dice = 2*bin_tp / max(2*bin_tp + bin_fp + bin_fn, 1e-12)
    result = {
        "binary_dice": float(binary_dice),
        "macro_fg_dice": macro,
        "gmean_fg_dice": gmean,
        "checkpoint_score": score,
        "dices": dices,
        "ious": ious,
        "cm": cm.numpy(),
    }
    if return_rows:
        result["rows"] = rows
    return result


def calibrate_on_validation(cached):
    best = None
    for thr in BIN_THRESHOLDS:
        for lam in PRIOR_LAMBDAS:
            m = score_cached(cached, thr, lam, return_rows=False)
            rec = {"bin_threshold": thr, "prior_lambda": lam, **{k:v for k,v in m.items() if k != "cm"}}
            if best is None or rec["checkpoint_score"] > best["checkpoint_score"]:
                best = rec
    return best


In [ ]:
# 9. Fold training loop

def make_loaders(fold):
    mf = make_fold_manifest(fold)
    tr = mf[mf.split == "train"].reset_index(drop=True)
    va = mf[mf.split == "val"].reset_index(drop=True)
    te = mf[mf.split == "test"].reset_index(drop=True)

    train_ds = EBHIWeakOrdinalDataset(tr, train=True, paired_stain=True)
    val_ds = EBHIWeakOrdinalDataset(va, train=False, paired_stain=False)
    test_ds = EBHIWeakOrdinalDataset(te, train=False, paired_stain=False)

    sampler = build_sampler(tr)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS,
                              pin_memory=PIN_MEMORY, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    return mf, tr, va, te, train_loader, val_loader, test_loader


def train_fold(fold):
    print("\n" + "="*90)
    print("FOLD", fold)
    mf, tr, va, te, train_loader, val_loader, test_loader = make_loaders(fold)
    grade_weights = fold_grade_ce_weights(tr)
    print("train/val/test:", len(tr), len(va), len(te), "| grade CE weights:", grade_weights.detach().cpu().numpy())

    model = MASOR21Net().to(DEVICE)
    teacher = copy.deepcopy(model).to(DEVICE).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    audit_optimizer_contains_ordinal_params(model, optimizer)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

    best_score = -np.inf
    best_path = CHECKPOINT_ROOT / f"fold_{fold}_best.pt"
    history = []
    bad_epochs = 0
    gradient_audit_done = False

    for epoch in range(EPOCHS):
        model.train()
        sums = defaultdict(float)
        n_batches = 0

        for batch in train_loader:
            clean = batch["clean"].to(DEVICE, non_blocking=True)
            stain = batch["stain"].to(DEVICE, non_blocking=True)
            batch["mask"] = batch["mask"].to(DEVICE, non_blocking=True)
            batch["grade"] = batch["grade"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                out_c = model(clean)
                out_s = model(stain)
                total, parts = compute_total_loss(model, out_c, out_s, batch, epoch, grade_weights, teacher=teacher)

            scaler.scale(total).backward()

            if not gradient_audit_done:
                # unscale before reading gradient norms
                scaler.unscale_(optimizer)
                grad_report = {
                    "ordinal_axis": float(model.ordinal_axis.grad.norm().detach().cpu()) if model.ordinal_axis.grad is not None else 0.0,
                    "gap_raw": float(model.gap_raw.grad.norm().detach().cpu()) if model.gap_raw.grad is not None else 0.0,
                    "proto_residual": float(model.proto_residual.grad.norm().detach().cpu()) if model.proto_residual.grad is not None else 0.0,
                }
                print("Ordinal gradient audit:", grad_report)
                assert grad_report["ordinal_axis"] > 0, "Ordinal axis received no gradient."
                assert grad_report["gap_raw"] > 0, "Prototype gaps received no gradient."
                gradient_audit_done = True
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

            update_ema(teacher, model)
            for k, v in parts.items():
                sums[k] += float(v.detach().cpu())
            n_batches += 1

        scheduler.step()

        # Fast validation without TTA; fixed inference settings for checkpoint selection.
        val_cached = collect_logits(model, val_loader, tta=False)
        vm = score_cached(val_cached, DEFAULT_BIN_THRESHOLD, DEFAULT_PRIOR_LAMBDA)
        rec = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            **{f"train_{k}": v/max(n_batches,1) for k,v in sums.items()},
            "val_binary_dice": vm["binary_dice"],
            "val_macro_fg_dice": vm["macro_fg_dice"],
            "val_gmean_fg_dice": vm["gmean_fg_dice"],
            "val_score": vm["checkpoint_score"],
            **{f"val_dice_{k}": v for k,v in vm["dices"].items()},
        }
        with torch.no_grad():
            _, centers, gaps = model.prototypes()
            rec.update({f"center_{i}": float(v) for i,v in enumerate(centers.detach().cpu())})
            rec.update({f"gap_{i}": float(v) for i,v in enumerate(gaps.detach().cpu())})
        history.append(rec)

        print(
            f"epoch {epoch:02d} | loss {rec['train_total']:.4f} | "
            f"val macro {vm['macro_fg_dice']:.4f} | gmean {vm['gmean_fg_dice']:.4f} | "
            f"bin {vm['binary_dice']:.4f} | HGD {vm['dices']['high_grade']:.4f}"
        )

        if vm["checkpoint_score"] > best_score + 1e-5:
            best_score = vm["checkpoint_score"]
            bad_epochs = 0
            torch.save({
                "model": model.state_dict(),
                "epoch": epoch,
                "val_score": best_score,
                "fold": fold,
                "master_manifest_sha256": MASTER_SHA256,
            }, best_path)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("Early stopping.")
                break

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(ARTIFACT_ROOT / f"fold_{fold}_history.csv", index=False)

    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    model.eval()

    # Validation-only inference calibration, now with the same TTA policy used for test.
    val_cached = collect_logits(model, val_loader, tta=TTA_ENABLE)
    calib = calibrate_on_validation(val_cached)
    print("Calibrated on inner validation:", {k: calib[k] for k in ["bin_threshold","prior_lambda","macro_fg_dice","gmean_fg_dice","binary_dice"]})

    test_cached = collect_logits(model, test_loader, tta=TTA_ENABLE)
    tm = score_cached(test_cached, calib["bin_threshold"], calib["prior_lambda"], return_rows=True)

    # Save prototype state for proof that N1 actually learned.
    with torch.no_grad():
        protos, centers, gaps = model.prototypes()
    proto_df = pd.DataFrame(protos.detach().cpu().numpy(), index=GRADE_NAMES)
    proto_df.to_csv(ARTIFACT_ROOT / f"fold_{fold}_learned_prototypes.csv")
    pd.DataFrame({"grade": GRADE_NAMES, "center": centers.detach().cpu().numpy()}).to_csv(
        ARTIFACT_ROOT / f"fold_{fold}_prototype_centers.csv", index=False
    )

    fold_result = {
        "fold": fold,
        "n_test_images": len(te),
        "n_test_groups": int(te.group_key.nunique()),
        "best_epoch": int(ckpt["epoch"]),
        "bin_threshold": float(calib["bin_threshold"]),
        "prior_lambda": float(calib["prior_lambda"]),
        "binary_dice": tm["binary_dice"],
        "macro_fg_dice": tm["macro_fg_dice"],
        "gmean_fg_dice": tm["gmean_fg_dice"],
        **{f"dice_{k}": float(v) for k,v in tm["dices"].items()},
        **{f"iou_{k}": float(v) for k,v in tm["ious"].items()},
        "checkpoint_sha256": hashlib.sha256(best_path.read_bytes()).hexdigest(),
        "master_manifest_sha256": MASTER_SHA256,
    }
    with open(ARTIFACT_ROOT / f"fold_{fold}_result.json", "w") as f:
        json.dump(fold_result, f, indent=2)

    rows = pd.DataFrame(tm["rows"])
    rows["fold"] = fold
    rows.to_csv(ARTIFACT_ROOT / f"fold_{fold}_test_per_image.csv", index=False)
    pd.DataFrame(tm["cm"], index=SEM_CLASS_NAMES, columns=SEM_CLASS_NAMES).to_csv(
        ARTIFACT_ROOT / f"fold_{fold}_confusion_counts.csv"
    )
    return fold_result, rows


In [ ]:
# 10. Run five folds
all_fold_results = []
all_oof_rows = []

for fold in FOLDS_TO_RUN:
    fr, rows = train_fold(fold)
    all_fold_results.append(fr)
    all_oof_rows.append(rows)

fold_results = pd.DataFrame(all_fold_results).sort_values("fold")
oof_rows = pd.concat(all_oof_rows, ignore_index=True)
fold_results.to_csv(ARTIFACT_ROOT / "fold_comparison.csv", index=False)
oof_rows.to_csv(ARTIFACT_ROOT / "OOF_per_image.csv", index=False)
display(fold_results)


In [ ]:
# 11. OOF aggregation from saved per-image confusion components

def aggregate_rows_to_metrics(df):
    cm = np.zeros((len(SEM_CLASS_NAMES), len(SEM_CLASS_NAMES)), dtype=np.int64)
    # Reconstruct a confusion matrix from per-class tp/fp/fn is underdetermined, so use Dice directly from aggregate one-vs-rest counts.
    dices = {}
    ious = {}
    for name in SEM_CLASS_NAMES:
        tp = df[f"tp_{name}"].sum()
        fp = df[f"fp_{name}"].sum()
        fn = df[f"fn_{name}"].sum()
        dices[name] = float(2*tp / max(2*tp + fp + fn, 1e-12))
        ious[name] = float(tp / max(tp + fp + fn, 1e-12))
    fg = np.array([dices[n] for n in GRADE_NAMES])
    macro = float(fg.mean())
    gmean = float(np.exp(np.log(np.clip(fg, 1e-8, 1)).mean()))
    btp, bfp, bfn = df.bin_tp.sum(), df.bin_fp.sum(), df.bin_fn.sum()
    bd = float(2*btp / max(2*btp + bfp + bfn, 1e-12))
    return {"binary_dice": bd, "macro_fg_dice": macro, "gmean_fg_dice": gmean, "dices": dices, "ious": ious}

oof_metrics = aggregate_rows_to_metrics(oof_rows)
headline = {
    "n_images": int(len(oof_rows)),
    "n_groups": int(oof_rows.group_key.nunique()),
    "binary_localisation_dice": oof_metrics["binary_dice"],
    "macro_fg_dice": oof_metrics["macro_fg_dice"],
    "gmean_fg_dice": oof_metrics["gmean_fg_dice"],
    "per_class_dice": oof_metrics["dices"],
    "per_class_iou": oof_metrics["ious"],
}
with open(ARTIFACT_ROOT / "OOF_headline_results.json", "w") as f:
    json.dump(headline, f, indent=2)
print(json.dumps(headline, indent=2))


In [ ]:
# 12. Group bootstrap CIs (resample source groups, not images)

def bootstrap_group_metrics(df, n_boot=BOOTSTRAP_ITERS, seed=SEED+9999):
    rng = np.random.default_rng(seed)
    metric_cols = []
    for name in SEM_CLASS_NAMES:
        metric_cols += [f"tp_{name}", f"fp_{name}", f"fn_{name}"]
    metric_cols += ["bin_tp", "bin_fp", "bin_fn"]
    group_sum = df.groupby("group_key")[metric_cols].sum().reset_index(drop=True)
    arr = group_sum[metric_cols].to_numpy(dtype=np.float64)
    n_groups = len(group_sum)
    records = []
    for _ in range(n_boot):
        sel = rng.integers(0, n_groups, size=n_groups)
        s = arr[sel].sum(axis=0)
        pos = {c:i for i,c in enumerate(metric_cols)}
        dices = {}
        for name in SEM_CLASS_NAMES:
            tp = s[pos[f"tp_{name}"]]; fp = s[pos[f"fp_{name}"]]; fn = s[pos[f"fn_{name}"]]
            dices[name] = 2*tp / max(2*tp + fp + fn, 1e-12)
        fg = np.array([dices[n] for n in GRADE_NAMES])
        btp, bfp, bfn = s[pos["bin_tp"]], s[pos["bin_fp"]], s[pos["bin_fn"]]
        rec = {
            "binary_dice": 2*btp / max(2*btp + bfp + bfn, 1e-12),
            "macro_fg_dice": fg.mean(),
            "gmean_fg_dice": np.exp(np.log(np.clip(fg, 1e-8, 1)).mean()),
        }
        rec.update({f"dice_{k}": v for k,v in dices.items()})
        records.append(rec)
    return pd.DataFrame(records)

boot = bootstrap_group_metrics(oof_rows)
boot.to_csv(ARTIFACT_ROOT / "OOF_group_bootstrap_distributions.csv", index=False)
ci_rows = []
for col in boot.columns:
    ci_rows.append({
        "metric": col,
        "ci_lo": float(boot[col].quantile(0.025)),
        "ci_hi": float(boot[col].quantile(0.975)),
        "bootstrap_mean": float(boot[col].mean()),
        "n_boot": len(boot),
    })
ci = pd.DataFrame(ci_rows)
ci.to_csv(ARTIFACT_ROOT / "OOF_bootstrap_ci.csv", index=False)
display(ci)


In [ ]:
# 13. Fold stability plot
plt.figure(figsize=(10,6))
plt.plot(fold_results.fold, fold_results.binary_dice, marker="o", label="binary Dice")
plt.plot(fold_results.fold, fold_results.macro_fg_dice, marker="o", label="macro FG Dice")
plt.plot(fold_results.fold, fold_results.dice_high_grade, marker="o", label="HGD Dice")
plt.xlabel("Outer fold")
plt.ylabel("Held-out score")
plt.title("MASOR-2.1 outer-fold stability")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "OOF_fold_stability.png", dpi=180)
plt.show()


## Optional stain robustness audit

This audit reloads the frozen best checkpoint for each outer fold and compares clean predictions against new random H&E perturbations. Crucially, **no stain normalisation is used before inference**.


In [ ]:
# 14. Optional stain robustness audit (can be skipped to save runtime)
RUN_STAIN_ROBUSTNESS = True

@torch.no_grad()
def predict_semantic_single_view(model, x, threshold, prior_lambda):
    b, g = infer_logits(model, x, tta=False)
    sem, bin_prob = semantic_from_logits(b, g, (IMAGE_SIZE, IMAGE_SIZE), threshold, prior_lambda)
    return sem, (bin_prob >= threshold).long()


def dice_binary_tensors(a, b):
    a = a.bool(); b = b.bool()
    inter = (a & b).sum().item()
    den = a.sum().item() + b.sum().item()
    return 2*inter / max(den, 1)

stain_rows = []
if RUN_STAIN_ROBUSTNESS:
    for _, fr in fold_results.iterrows():
        fold = int(fr.fold)
        mf = make_fold_manifest(fold)
        te = mf[mf.split == "test"].reset_index(drop=True)
        ckpt = torch.load(CHECKPOINT_ROOT / f"fold_{fold}_best.pt", map_location=DEVICE)
        model = MASOR21Net().to(DEVICE)
        model.load_state_dict(ckpt["model"])
        model.eval()

        for rep in range(STAIN_EVAL_REPEATS):
            np.random.seed(SEED + fold*100 + rep)
            random.seed(SEED + fold*100 + rep)
            ds = EBHIWeakOrdinalDataset(te, train=False, paired_stain=True)
            loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
            for batch in loader:
                clean = batch["clean"].to(DEVICE)
                stain = batch["stain"].to(DEVICE)
                mask = batch["mask"][:,0].long()
                csem, cbin = predict_semantic_single_view(model, clean, float(fr.bin_threshold), float(fr.prior_lambda))
                ssem, sbin = predict_semantic_single_view(model, stain, float(fr.bin_threshold), float(fr.prior_lambda))
                for i in range(len(batch["sample_id"])):
                    gt = mask[i]
                    clean_d = dice_binary_tensors(cbin[i].cpu(), gt)
                    stain_d = dice_binary_tensors(sbin[i].cpu(), gt)
                    fg = gt.bool()
                    class_stability = float((csem[i].cpu()[fg] == ssem[i].cpu()[fg]).float().mean()) if fg.any() else 1.0
                    stain_rows.append({
                        "fold": fold,
                        "repeat": rep,
                        "sample_id": batch["sample_id"][i],
                        "group_key": batch["group_key"][i],
                        "fgbg_stability": dice_binary_tensors(cbin[i].cpu(), sbin[i].cpu()),
                        "fg_class_stability": class_stability,
                        "clean_fg_dice_vs_gt": clean_d,
                        "stain_fg_dice_vs_gt": stain_d,
                        "dice_drop": clean_d - stain_d,
                    })

    stain_df = pd.DataFrame(stain_rows)
    stain_df.to_csv(ARTIFACT_ROOT / "OOF_stain_robustness.csv", index=False)
    stain_headline = {
        "mean_fgbg_stability": float(stain_df.fgbg_stability.mean()),
        "mean_fg_class_stability": float(stain_df.fg_class_stability.mean()),
        "mean_clean_fg_dice_vs_gt": float(stain_df.clean_fg_dice_vs_gt.mean()),
        "mean_stain_fg_dice_vs_gt": float(stain_df.stain_fg_dice_vs_gt.mean()),
        "mean_dice_drop": float(stain_df.dice_drop.mean()),
        "n_repeats_per_image": STAIN_EVAL_REPEATS,
        "stain_normalisation_used": False,
    }
    with open(ARTIFACT_ROOT / "OOF_stain_headline.json", "w") as f:
        json.dump(stain_headline, f, indent=2)
    print(json.dumps(stain_headline, indent=2))


In [ ]:
# 15. Prototype audit: prove that the ordinal geometry genuinely moved
proto_rows = []
for fold in FOLDS_TO_RUN:
    p = pd.read_csv(ARTIFACT_ROOT / f"fold_{fold}_prototype_centers.csv")
    p["fold"] = fold
    proto_rows.append(p)
proto_centers = pd.concat(proto_rows, ignore_index=True)
proto_centers.to_csv(ARTIFACT_ROOT / "learned_prototype_centers_all_folds.csv", index=False)

pivot = proto_centers.pivot(index="grade", columns="fold", values="center").reindex(GRADE_NAMES)
display(pivot)
print("Across-fold centre SD:")
print(pivot.std(axis=1))

# Fail loudly if the prototypes remained exactly at one identical configuration across every fold.
if np.allclose(pivot.std(axis=1).to_numpy(), 0.0, atol=1e-10):
    raise RuntimeError("Ordinal prototypes are identical across all folds; investigate optimisation before publication.")


In [ ]:
# 16. Compact results summary
summary_rows = []
for _, r in fold_results.iterrows():
    summary_rows.append({
        "fold": int(r.fold),
        "binary_dice": r.binary_dice,
        "normal": r.dice_normal_gland,
        "LGD": r.dice_low_grade,
        "HGD": r.dice_high_grade,
        "ADC": r.dice_adk_invasion,
        "macro_fg": r.macro_fg_dice,
        "gmean_fg": r.gmean_fg_dice,
        "bin_thr": r.bin_threshold,
        "prior_lambda": r.prior_lambda,
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
print("\nOOF headline:")
print(json.dumps(headline, indent=2))


## What to compare next

Once this clean full model runs successfully, do **not** redesign it again before the ablations. The next controlled experiments should reuse the exact same folds and differ by one switch at a time:

1. **No ordinal prototype geometry** — replace prototype logits with a plain 1×1 grade classifier.
2. **No feature-level stain InfoNCE** — set `W_INFO_NCE = 0`.
3. **No EMA SoftGrade** — set `W_SOFTGRADE = 0` and retain the weak pixel term.
4. **No HGD neighbour margin** — set `W_HGD_MARGIN = 0`.
5. **No image prior fusion** — force `PRIOR_LAMBDAS = [0.0]`.

The primary endpoint should remain **macro foreground Dice**, with HGD Dice and group-bootstrap confidence intervals reported explicitly. Binary localisation Dice is a co-primary sanity metric because the proposed method should not gain grade performance by destroying foreground localisation.


In [ ]:
# ============================================================
# MASOR-2.1 — RANDOM HELD-OUT EXAMPLE FROM EACH CLASS
#
# Shows:
# ORIGINAL | WEAK SEMANTIC GT | MASOR PREDICTION | OVERLAY
#
# Each image is evaluated using the checkpoint from the outer
# fold in which that image was held out.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from PIL import Image


# ============================================================
# 0. RECOVER SAVED EXPERIMENT PATHS IF KERNEL WAS RESTARTED
# ============================================================

if "WORK_ROOT" not in globals():

    candidates = list(
        Path("/kaggle/working").rglob("fold_0_best.pt")
    )

    if len(candidates) == 0:
        raise FileNotFoundError(
            "Could not find fold_0_best.pt anywhere under /kaggle/working."
        )

    # Prefer the MASOR-2.1 folder if present
    preferred = [
        p for p in candidates
        if "masor21" in str(p).lower()
    ]

    ckpt0 = preferred[0] if preferred else candidates[0]

    CHECKPOINT_ROOT = ckpt0.parent
    WORK_ROOT = CHECKPOINT_ROOT.parent
    ARTIFACT_ROOT = WORK_ROOT / "artifacts"

else:
    WORK_ROOT = Path(WORK_ROOT)
    ARTIFACT_ROOT = WORK_ROOT / "artifacts"
    CHECKPOINT_ROOT = WORK_ROOT / "checkpoints"


print("Experiment:", WORK_ROOT)


# ============================================================
# 1. LOAD SAVED TABLES
# ============================================================

master = pd.read_csv(
    WORK_ROOT / "master_manifest.csv"
)

fold_results = pd.read_csv(
    ARTIFACT_ROOT / "fold_comparison.csv"
)


# ============================================================
# 2. CHECK THAT MODEL DEFINITIONS EXIST IN MEMORY
# ============================================================

required = [
    "MASOR21Net",
    "EBHIWeakOrdinalDataset",
    "infer_logits",
    "semantic_from_logits",
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "These notebook definitions are not currently loaded:\n"
        + "\n".join(f"  - {x}" for x in missing)
        + "\n\nRerun the model/inference definition cells only — "
          "you DO NOT need to retrain."
    )


# ============================================================
# 3. CLASS DEFINITIONS
# ============================================================

grades = [
    "normal_gland",
    "low_grade",
    "high_grade",
    "adk_invasion",
]

pretty_names = {
    "normal_gland": "Normal gland",
    "low_grade": "Low-grade IN",
    "high_grade": "High-grade IN",
    "adk_invasion": "Adenocarcinoma",
}

# Semantic classes:
# 0 background
# 1 normal
# 2 LGD
# 3 HGD
# 4 ADC

semantic_colors = [
    "black",
    "limegreen",
    "gold",
    "darkorange",
    "red",
]

cmap = ListedColormap(semantic_colors)

semantic_labels = [
    "Background",
    "Normal",
    "LGD",
    "HGD",
    "ADC",
]


# ============================================================
# 4. RANDOM SAMPLE — ONE IMAGE PER CLASS
# ============================================================

rng = np.random.default_rng()

chosen = []

for cls in grades:

    subset = master[
        master["target_class"] == cls
    ]

    if len(subset) == 0:
        raise ValueError(
            f"No samples found for {cls}"
        )

    idx = int(
        rng.integers(
            0,
            len(subset)
        )
    )

    chosen.append(
        subset.iloc[idx]
    )


# ============================================================
# 5. HELPER: DICE
# ============================================================

def dice_np(a, b):

    a = a.astype(bool)
    b = b.astype(bool)

    intersection = np.logical_and(
        a,
        b,
    ).sum()

    denom = (
        a.sum()
        + b.sum()
    )

    if denom == 0:
        return 1.0

    return (
        2.0
        * intersection
        / denom
    )


# ============================================================
# 6. RUN EACH RANDOM IMAGE THROUGH ITS HELD-OUT MODEL
# ============================================================

results = []

for row in chosen:

    cls = row["target_class"]

    fold = int(
        row["outer_fold"]
    )

    # --------------------------------------------------------
    # Load corresponding held-out checkpoint
    # --------------------------------------------------------

    checkpoint_path = (
        CHECKPOINT_ROOT
        / f"fold_{fold}_best.pt"
    )

    ckpt = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model = MASOR21Net().to(DEVICE)

    model.load_state_dict(
        ckpt["model"]
    )

    model.eval()

    # --------------------------------------------------------
    # Fold-specific calibration learned ONLY on inner val
    # --------------------------------------------------------

    fr = fold_results[
        fold_results["fold"].astype(int)
        == fold
    ].iloc[0]

    bin_threshold = float(
        fr["bin_threshold"]
    )

    prior_lambda = float(
        fr["prior_lambda"]
    )

    # --------------------------------------------------------
    # Build one-image dataset
    # --------------------------------------------------------

    one_df = pd.DataFrame(
        [row]
    )

    ds = EBHIWeakOrdinalDataset(
        one_df,
        train=False,
        paired_stain=False,
    )

    sample = ds[0]

    x = (
        sample["clean"]
        .unsqueeze(0)
        .to(DEVICE)
    )

    # --------------------------------------------------------
    # Raw image
    # --------------------------------------------------------

    rgb = np.asarray(
        Image.open(
            row["image"]
        )
        .convert("RGB")
        .resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),
            Image.Resampling.BILINEAR,
        )
    )

    # --------------------------------------------------------
    # Real EBHI binary mask
    # --------------------------------------------------------

    gt_mask = np.asarray(
        Image.open(
            row["mask"]
        )
        .convert("L")
        .resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),
            Image.Resampling.NEAREST,
        )
    )

    gt_mask = (
        gt_mask
        >= SOURCE_MASK_THRESHOLD
    )

    # --------------------------------------------------------
    # Inference
    # --------------------------------------------------------

    with torch.no_grad():

        bin_logits, grade_logits = infer_logits(
            model,
            x,
            tta=TTA_ENABLE,
        )

        pred_sem, bin_prob = semantic_from_logits(
            bin_logits,
            grade_logits,
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),
            bin_threshold,
            prior_lambda,
        )

    pred_sem = (
        pred_sem[0]
        .detach()
        .cpu()
        .numpy()
    )

    bin_prob = (
        bin_prob[0, 0]
        .detach()
        .cpu()
        .numpy()
    )

    pred_fg = (
        pred_sem > 0
    )

    # --------------------------------------------------------
    # Weak semantic reference
    #
    # EBHI gives binary GT + image-level grade.
    # Therefore the reference semantic map paints the
    # image-level diagnosis over the real foreground mask.
    # --------------------------------------------------------

    true_grade_idx = int(
        row["grade_idx"]
    )

    true_semantic_idx = (
        true_grade_idx + 1
    )

    gt_sem = np.zeros(
        (
            IMAGE_SIZE,
            IMAGE_SIZE,
        ),
        dtype=np.uint8,
    )

    gt_sem[
        gt_mask
    ] = true_semantic_idx

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    binary_dice = dice_np(
        pred_fg,
        gt_mask,
    )

    true_grade_pred_mask = (
        pred_sem
        == true_semantic_idx
    )

    grade_dice = dice_np(
        true_grade_pred_mask,
        gt_mask,
    )

    # --------------------------------------------------------
    # Predicted grade composition
    # --------------------------------------------------------

    composition = {}

    n_fg = max(
        int(pred_fg.sum()),
        1,
    )

    for sem_idx, name in enumerate(
        semantic_labels[1:],
        start=1,
    ):

        composition[name] = (
            (pred_sem == sem_idx).sum()
            / n_fg
        )

    results.append(
        {
            "row": row,
            "fold": fold,
            "rgb": rgb,
            "gt_sem": gt_sem,
            "pred_sem": pred_sem,
            "binary_dice": binary_dice,
            "grade_dice": grade_dice,
            "composition": composition,
            "bin_threshold": bin_threshold,
            "prior_lambda": prior_lambda,
        }
    )


# ============================================================
# 7. PLOT — ONE ROW PER TRUE CLASS
# ============================================================

fig, axes = plt.subplots(
    nrows=4,
    ncols=4,
    figsize=(18, 18),
)

for r, res in enumerate(results):

    row = res["row"]

    true_name = pretty_names[
        row["target_class"]
    ]

    # --------------------------------------------------------
    # ORIGINAL
    # --------------------------------------------------------

    axes[r, 0].imshow(
        res["rgb"]
    )

    axes[r, 0].set_title(
        f"{true_name}\n"
        f"Original — fold {res['fold']}",
        fontsize=12,
        fontweight="bold",
    )

    # --------------------------------------------------------
    # WEAK GT SEMANTIC MAP
    # --------------------------------------------------------

    axes[r, 1].imshow(
        res["gt_sem"],
        cmap=cmap,
        vmin=0,
        vmax=4,
        interpolation="nearest",
    )

    axes[r, 1].set_title(
        "Weak semantic reference",
        fontsize=12,
    )

    # --------------------------------------------------------
    # PREDICTED SEMANTIC MAP
    # --------------------------------------------------------

    axes[r, 2].imshow(
        res["pred_sem"],
        cmap=cmap,
        vmin=0,
        vmax=4,
        interpolation="nearest",
    )

    axes[r, 2].set_title(
        "MASOR prediction\n"
        f"FG Dice={res['binary_dice']:.3f} | "
        f"Grade Dice={res['grade_dice']:.3f}",
        fontsize=12,
    )

    # --------------------------------------------------------
    # OVERLAY
    # --------------------------------------------------------

    axes[r, 3].imshow(
        res["rgb"]
    )

    overlay = np.ma.masked_where(
        res["pred_sem"] == 0,
        res["pred_sem"],
    )

    axes[r, 3].imshow(
        overlay,
        cmap=cmap,
        vmin=0,
        vmax=4,
        alpha=0.50,
        interpolation="nearest",
    )

    comp = res["composition"]

    axes[r, 3].set_title(
        "Prediction overlay\n"
        f"N {comp['Normal']*100:.1f}% | "
        f"L {comp['LGD']*100:.1f}% | "
        f"H {comp['HGD']*100:.1f}% | "
        f"A {comp['ADC']*100:.1f}%",
        fontsize=11,
    )

    for c in range(4):
        axes[r, c].axis("off")


# ============================================================
# 8. COLUMN LABELS
# ============================================================

axes[0, 0].text(
    0.5,
    1.18,
    "INPUT",
    transform=axes[0, 0].transAxes,
    ha="center",
    fontsize=15,
    fontweight="bold",
)

axes[0, 1].text(
    0.5,
    1.18,
    "REFERENCE",
    transform=axes[0, 1].transAxes,
    ha="center",
    fontsize=15,
    fontweight="bold",
)

axes[0, 2].text(
    0.5,
    1.18,
    "SEMANTIC SEGMENTATION",
    transform=axes[0, 2].transAxes,
    ha="center",
    fontsize=15,
    fontweight="bold",
)

axes[0, 3].text(
    0.5,
    1.18,
    "OVERLAY",
    transform=axes[0, 3].transAxes,
    ha="center",
    fontsize=15,
    fontweight="bold",
)


# ============================================================
# 9. LEGEND
# ============================================================

legend_handles = [

    Patch(
        facecolor=semantic_colors[i],
        label=semantic_labels[i],
    )

    for i in range(5)
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=5,
    fontsize=12,
    frameon=True,
)


fig.suptitle(
    "MASOR-2.1 — Random Held-Out Example From Each EBHI Class",
    fontsize=18,
    fontweight="bold",
    y=0.995,
)

plt.tight_layout(
    rect=[
        0,
        0.05,
        1,
        0.98,
    ]
)

plt.show()


# ============================================================
# 10. PRINT NUMERICAL SUMMARY
# ============================================================

summary = []

for res in results:

    row = res["row"]

    summary.append(
        {
            "true_class": pretty_names[
                row["target_class"]
            ],
            "sample_id": row["sample_id"],
            "outer_fold": res["fold"],
            "binary_dice": res["binary_dice"],
            "true_grade_dice": res["grade_dice"],
            "pred_normal_%":
                100 * res["composition"]["Normal"],
            "pred_LGD_%":
                100 * res["composition"]["LGD"],
            "pred_HGD_%":
                100 * res["composition"]["HGD"],
            "pred_ADC_%":
                100 * res["composition"]["ADC"],
        }
    )

summary_df = pd.DataFrame(summary)

display(
    summary_df.style.format(
        {
            "binary_dice": "{:.3f}",
            "true_grade_dice": "{:.3f}",
            "pred_normal_%": "{:.1f}",
            "pred_LGD_%": "{:.1f}",
            "pred_HGD_%": "{:.1f}",
            "pred_ADC_%": "{:.1f}",
        }
    )
)

In [ ]:
# ==============================================================
# MASOR-2.1 FORENSIC AUDIT A
# Files, CV leakage, checkpoints, histories, prototypes
# ==============================================================

from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import json

# --------------------------------------------------------------
# Recover experiment
# --------------------------------------------------------------

if "WORK_ROOT" not in globals():
    hits = list(Path("/kaggle/working").rglob("fold_0_best.pt"))
    if not hits:
        raise FileNotFoundError("No fold_0_best.pt under /kaggle/working")

    preferred = [p for p in hits if "masor21" in str(p).lower()]
    ckpt0 = preferred[0] if preferred else hits[0]

    CHECKPOINT_ROOT = ckpt0.parent
    WORK_ROOT = CHECKPOINT_ROOT.parent
    ARTIFACT_ROOT = WORK_ROOT / "artifacts"

WORK_ROOT = Path(WORK_ROOT)
ARTIFACT_ROOT = Path(ARTIFACT_ROOT)
CHECKPOINT_ROOT = Path(CHECKPOINT_ROOT)

print("Experiment:", WORK_ROOT)
print()

master = pd.read_csv(WORK_ROOT / "master_manifest.csv")
fold_results = pd.read_csv(ARTIFACT_ROOT / "fold_comparison.csv")

# --------------------------------------------------------------
# 1. Basic result sanity
# --------------------------------------------------------------

metric_cols = [
    "binary_dice",
    "macro_fg_dice",
    "gmean_fg_dice",
    "dice_normal_gland",
    "dice_low_grade",
    "dice_high_grade",
    "dice_adk_invasion",
]

print("=" * 80)
print("FOLD STABILITY")
print("=" * 80)

summary = []

for col in metric_cols:
    x = fold_results[col].astype(float)

    summary.append({
        "metric": col,
        "mean": x.mean(),
        "std": x.std(ddof=1),
        "min": x.min(),
        "max": x.max(),
        "range": x.max() - x.min(),
        "CV_%": 100 * x.std(ddof=1) / max(x.mean(), 1e-12),
    })

summary_df = pd.DataFrame(summary)

display(
    summary_df.style.format({
        "mean": "{:.4f}",
        "std": "{:.4f}",
        "min": "{:.4f}",
        "max": "{:.4f}",
        "range": "{:.4f}",
        "CV_%": "{:.1f}",
    })
)

# --------------------------------------------------------------
# 2. Calibration-edge audit
# --------------------------------------------------------------

print("\n" + "=" * 80)
print("CALIBRATION AUDIT")
print("=" * 80)

display(
    fold_results[
        [
            "fold",
            "best_epoch",
            "bin_threshold",
            "prior_lambda",
            "binary_dice",
            "macro_fg_dice",
            "dice_high_grade",
        ]
    ]
)

if "PRIOR_LAMBDAS" in globals():
    lambda_max = max(PRIOR_LAMBDAS)
else:
    lambda_max = 0.75

n_at_max = int(
    np.isclose(
        fold_results["prior_lambda"].astype(float),
        lambda_max,
    ).sum()
)

print(
    f"\nprior λ at upper search boundary: "
    f"{n_at_max}/{len(fold_results)} folds"
)

if n_at_max == len(fold_results):
    print(
        "⚠️ RED FLAG TO INTERPRET: every fold selected the maximum allowed λ."
    )

# --------------------------------------------------------------
# 3. SHA / checkpoint integrity
# --------------------------------------------------------------

print("\n" + "=" * 80)
print("CHECKPOINT / MANIFEST INTEGRITY")
print("=" * 80)

manifest_shas = fold_results["master_manifest_sha256"].unique()

print("Unique manifest SHA256 values:", len(manifest_shas))

assert len(manifest_shas) == 1, (
    "Different folds appear to have used different master manifests."
)

checkpoint_shas = fold_results["checkpoint_sha256"].tolist()

print("Unique checkpoint SHA256 values:", len(set(checkpoint_shas)))

assert len(set(checkpoint_shas)) == len(checkpoint_shas), (
    "Two folds unexpectedly have identical checkpoint hashes."
)

for fold in fold_results["fold"].astype(int):

    p = CHECKPOINT_ROOT / f"fold_{fold}_best.pt"

    assert p.exists(), f"Missing checkpoint: {p}"

print("✓ All five checkpoints present")
print("✓ Same master manifest used for all folds")
print("✓ Checkpoints are distinct")

# --------------------------------------------------------------
# 4. Reconstruct split leakage if functions are available
# --------------------------------------------------------------

print("\n" + "=" * 80)
print("GROUP / SAMPLE LEAKAGE AUDIT")
print("=" * 80)

if "make_fold_manifest" not in globals():

    print(
        "make_fold_manifest() is not in memory.\n"
        "Rerun the split-definition cell if you want the full leakage audit."
    )

else:

    test_occurrences = Counter()
    all_good = True

    for fold in sorted(fold_results["fold"].astype(int)):

        mf = make_fold_manifest(fold)

        tr = mf[mf["split"] == "train"]
        va = mf[mf["split"] == "val"]
        te = mf[mf["split"] == "test"]

        tr_g = set(tr["group_key"])
        va_g = set(va["group_key"])
        te_g = set(te["group_key"])

        overlap_tv = tr_g & va_g
        overlap_tt = tr_g & te_g
        overlap_vt = va_g & te_g

        print(
            f"fold {fold}: "
            f"train={len(tr)} val={len(va)} test={len(te)} | "
            f"group overlaps = "
            f"{len(overlap_tv)}/{len(overlap_tt)}/{len(overlap_vt)}"
        )

        if overlap_tv or overlap_tt or overlap_vt:
            all_good = False

        # Also check exact file paths
        assert not (
            set(tr["image"])
            & set(te["image"])
        )

        assert not (
            set(va["image"])
            & set(te["image"])
        )

        for sid in te["sample_id"]:
            test_occurrences[sid] += 1

    assert all_good, "GROUP LEAKAGE FOUND."

    counts = np.array(
        [test_occurrences[sid] for sid in master["sample_id"]]
    )

    print("\nTest occurrence distribution:")
    print(pd.Series(counts).value_counts().sort_index())

    assert np.all(counts == 1), (
        "Each image should occur in exactly one outer test fold."
    )

    print("\n✓ Every sample appears in outer test exactly once")
    print("✓ No train/val/test group overlap detected")

# --------------------------------------------------------------
# 5. Exact duplicate audit
# --------------------------------------------------------------

print("\n" + "=" * 80)
print("DUPLICATE AUDIT")
print("=" * 80)

if "source_image_sha256" in master.columns:

    dup = master[
        master.duplicated(
            "source_image_sha256",
            keep=False
        )
    ].copy()

    print("Images participating in exact duplicate sets:", len(dup))

    if len(dup):

        conflicting = (
            dup.groupby("source_image_sha256")["target_class"]
            .nunique()
        )

        conflicting = conflicting[
            conflicting > 1
        ]

        print(
            "Duplicate hashes carrying multiple grades:",
            len(conflicting),
        )

        assert len(conflicting) == 0, (
            "Exact duplicate image appears with conflicting class labels."
        )

        # Duplicates should have been merged into one group
        bad_groups = (
            dup.groupby("source_image_sha256")["group_key"]
            .nunique()
        )

        bad_groups = bad_groups[
            bad_groups > 1
        ]

        print(
            "Duplicate hashes split across group keys:",
            len(bad_groups),
        )

        assert len(bad_groups) == 0, (
            "Duplicate images can cross CV groups."
        )

        print("✓ Exact duplicates are group-locked")

else:
    print("No source_image_sha256 column found.")

# --------------------------------------------------------------
# 6. Training-history / best checkpoint audit
# --------------------------------------------------------------

print("\n" + "=" * 80)
print("CHECKPOINT-SELECTION AUDIT")
print("=" * 80)

hist_rows = []

for _, fr in fold_results.iterrows():

    fold = int(fr["fold"])

    hp = ARTIFACT_ROOT / f"fold_{fold}_history.csv"

    if not hp.exists():
        print("Missing:", hp)
        continue

    h = pd.read_csv(hp)

    if "val_score" not in h.columns:
        print(f"fold {fold}: no val_score column")
        continue

    best_idx = h["val_score"].astype(float).idxmax()

    hist_best_epoch = int(
        h.loc[best_idx, "epoch"]
        if "epoch" in h.columns
        else best_idx
    )

    reported_best = int(fr["best_epoch"])

    hist_rows.append({
        "fold": fold,
        "reported_best_epoch": reported_best,
        "history_best_epoch": hist_best_epoch,
        "best_val_score": float(h.loc[best_idx, "val_score"]),
        "n_epochs_run": len(h),
        "best_in_first_10": hist_best_epoch < 10,
    })

    if hist_best_epoch != reported_best:
        print(
            f"⚠️ fold {fold}: history says {hist_best_epoch}, "
            f"result says {reported_best}"
        )

hist_audit = pd.DataFrame(hist_rows)

display(hist_audit)

if len(hist_audit):
    assert np.all(
        hist_audit["reported_best_epoch"]
        == hist_audit["history_best_epoch"]
    ), "Saved checkpoint does not correspond to history maximum."

print("✓ Saved best epochs agree with validation histories")

# --------------------------------------------------------------
# 7. Prototype geometry
# --------------------------------------------------------------

print("\n" + "=" * 80)
print("PROTOTYPE AUDIT")
print("=" * 80)

prototype_rows = []

for fold in sorted(fold_results["fold"].astype(int)):

    p = (
        ARTIFACT_ROOT
        / f"fold_{fold}_prototype_centers.csv"
    )

    if not p.exists():
        continue

    pdf = pd.read_csv(p)

    centers = pdf["center"].astype(float).values

    diffs = np.diff(centers)

    assert np.all(diffs > 0), (
        f"Prototype ordering broken in fold {fold}"
    )

    rec = {"fold": fold}

    for grade, center in zip(
        pdf["grade"],
        centers,
    ):
        rec[f"center_{grade}"] = center

    for i, d in enumerate(diffs):
        rec[f"gap_{i}"] = d

    prototype_rows.append(rec)

proto_audit = pd.DataFrame(prototype_rows)

if len(proto_audit):
    display(proto_audit)

    print("\nCross-fold prototype SD:")
    display(
        proto_audit.drop(columns="fold")
        .std()
        .to_frame("SD")
    )

    print("✓ Prototype ordering intact in every fold")

print("\nFORENSIC AUDIT A COMPLETE")

In [ ]:
# =====================================================================
# MASOR-2.1 FORENSIC AUDIT B — FULL CORRECTED VERSION
# =====================================================================
#
# PURPOSE:
# Test whether MASOR's dense semantic performance genuinely comes from
# local ordinal morphology, or whether the image-level prior is doing
# most of the grading work.
#
# We compare:
#
#   λ = 0.00   -> pure LOCAL prototype grading
#   λ = 0.25
#   λ = 0.50
#   λ = 0.75   -> current selected value
#   λ = 1.00   -> diagnostic sensitivity only
#   λ = 1.50   -> diagnostic sensitivity only
#
# PLUS:
#
#   IMAGE-GRADE + PAINT FOREGROUND
#       Model predicts one image grade, then paints all predicted
#       foreground pixels with that grade.
#
#   ORACLE TRUE-GRADE PAINT
#       Uses true image label + MASOR foreground.
#       THIS IS NOT A MODEL RESULT.
#
# IMPORTANT:
# λ=1.0 and λ=1.5 are post-hoc diagnostic probes only.
# DO NOT use outer-test performance to tune the model.
#
# =====================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from IPython.display import display


# =====================================================================
# 0. RECOVER EXPERIMENT PATHS IF NEEDED
# =====================================================================

if "WORK_ROOT" not in globals():

    hits = list(
        Path("/kaggle/working").rglob("fold_0_best.pt")
    )

    if len(hits) == 0:
        raise FileNotFoundError(
            "Could not find fold_0_best.pt under /kaggle/working."
        )

    preferred = [
        p for p in hits
        if "masor21" in str(p).lower()
    ]

    ckpt0 = (
        preferred[0]
        if len(preferred)
        else hits[0]
    )

    CHECKPOINT_ROOT = ckpt0.parent
    WORK_ROOT = CHECKPOINT_ROOT.parent
    ARTIFACT_ROOT = WORK_ROOT / "artifacts"

else:

    WORK_ROOT = Path(WORK_ROOT)
    ARTIFACT_ROOT = Path(
        globals().get(
            "ARTIFACT_ROOT",
            WORK_ROOT / "artifacts"
        )
    )
    CHECKPOINT_ROOT = Path(
        globals().get(
            "CHECKPOINT_ROOT",
            WORK_ROOT / "checkpoints"
        )
    )


print("=" * 90)
print("MASOR-2.1 FORENSIC AUDIT B")
print("=" * 90)

print("WORK_ROOT      :", WORK_ROOT)
print("ARTIFACT_ROOT  :", ARTIFACT_ROOT)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print()


# =====================================================================
# 1. LOAD SAVED EXPERIMENT TABLES
# =====================================================================

master = pd.read_csv(
    WORK_ROOT / "master_manifest.csv"
)

fold_results = pd.read_csv(
    ARTIFACT_ROOT / "fold_comparison.csv"
)

print(
    f"Master manifest: {len(master):,} images | "
    f"{master['group_key'].nunique():,} groups"
)

print(
    f"Fold results:    {len(fold_results)} folds"
)

display(
    fold_results[
        [
            "fold",
            "best_epoch",
            "bin_threshold",
            "prior_lambda",
            "binary_dice",
            "macro_fg_dice",
            "dice_high_grade",
        ]
    ]
)


# =====================================================================
# 2. CHECK REQUIRED NOTEBOOK DEFINITIONS
# =====================================================================

required = [
    "MASOR21Net",
    "EBHIWeakOrdinalDataset",
    "make_fold_manifest",
    "collect_logits",
    "semantic_from_logits",
    "masked_lse_pool",
    "SEM_CLASS_NAMES",
    "GRADE_NAMES",
    "LSE_TAU",
    "DEVICE",
    "BATCH_SIZE",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "\nThese notebook definitions are not currently loaded:\n\n"
        + "\n".join(
            f"  - {x}"
            for x in missing
        )
        + "\n\nRerun the relevant definition cells only. "
          "DO NOT rerun training."
    )


# =====================================================================
# 3. SETTINGS
# =====================================================================

K = len(SEM_CLASS_NAMES)

# Include the original calibration values plus larger λ values
# strictly as diagnostic sensitivity checks.
LAMBDAS_TO_AUDIT = [
    0.00,
    0.25,
    0.50,
    0.75,
    1.00,
    1.50,
]

print("\nSemantic classes:")
for i, name in enumerate(SEM_CLASS_NAMES):
    print(f"  {i}: {name}")

print("\nλ values to audit:", LAMBDAS_TO_AUDIT)


# =====================================================================
# 4. CONFUSION-MATRIX HELPERS
# =====================================================================

def add_cm(cm, gt, pred, n_classes):

    """
    Add a batch of predictions to an integer confusion matrix.

    gt and pred must both be B x H x W.
    """

    assert gt.shape == pred.shape, (
        f"GT/pred shape mismatch: "
        f"{tuple(gt.shape)} vs {tuple(pred.shape)}"
    )

    gt = gt.long().reshape(-1)
    pred = pred.long().reshape(-1)

    valid = (
        (gt >= 0)
        & (gt < n_classes)
        & (pred >= 0)
        & (pred < n_classes)
    )

    gt = gt[valid]
    pred = pred[valid]

    idx = (
        gt * n_classes
        + pred
    )

    counts = torch.bincount(
        idx,
        minlength=n_classes * n_classes,
    )

    cm += (
        counts
        .reshape(
            n_classes,
            n_classes,
        )
        .cpu()
    )


def metrics_from_cm(cm):

    """
    Calculate Dice / IoU from pooled pixel confusion counts.
    """

    cm = cm.double()

    dices = {}
    ious = {}

    for c, name in enumerate(
        SEM_CLASS_NAMES
    ):

        tp = cm[c, c]

        fp = (
            cm[:, c].sum()
            - tp
        )

        fn = (
            cm[c, :].sum()
            - tp
        )

        dice_den = (
            2 * tp
            + fp
            + fn
        )

        iou_den = (
            tp
            + fp
            + fn
        )

        dice = (
            float(
                (2 * tp / dice_den).item()
            )
            if dice_den > 0
            else np.nan
        )

        iou = (
            float(
                (tp / iou_den).item()
            )
            if iou_den > 0
            else np.nan
        )

        dices[name] = dice
        ious[name] = iou

    fg_dices = np.array(
        [
            dices[name]
            for name in GRADE_NAMES
        ],
        dtype=float,
    )

    macro = float(
        np.nanmean(
            fg_dices
        )
    )

    valid_fg = fg_dices[
        np.isfinite(fg_dices)
    ]

    gmean = float(
        np.exp(
            np.log(
                np.clip(
                    valid_fg,
                    1e-12,
                    1.0,
                )
            ).mean()
        )
    )

    return {
        "macro_fg_dice": macro,
        "gmean_fg_dice": gmean,
        **{
            f"dice_{k}": v
            for k, v in dices.items()
        },
        **{
            f"iou_{k}": v
            for k, v in ious.items()
        },
    }


# =====================================================================
# 5. HOMOGENEITY HELPER
# =====================================================================

def image_homogeneity(sem):

    """
    Fraction of predicted foreground occupied by the dominant grade.

    1.0 = all predicted foreground is one grade.
    Lower = locally heterogeneous grade field.
    """

    assert sem.ndim == 2, (
        f"Expected HxW semantic map, got {sem.shape}"
    )

    fg = (
        sem > 0
    )

    if not fg.any():
        return np.nan

    vals = sem[fg]

    counts = torch.stack(
        [
            (vals == c).sum()
            for c in range(
                1,
                K,
            )
        ]
    ).float()

    return float(
        (
            counts.max()
            / counts.sum().clamp_min(1)
        ).item()
    )


# =====================================================================
# 6. GLOBAL ACCUMULATORS
# =====================================================================

global_cm = {
    lam: torch.zeros(
        (
            K,
            K,
        ),
        dtype=torch.long,
    )
    for lam in LAMBDAS_TO_AUDIT
}

paint_cm = torch.zeros(
    (
        K,
        K,
    ),
    dtype=torch.long,
)

oracle_paint_cm = torch.zeros(
    (
        K,
        K,
    ),
    dtype=torch.long,
)

raw_homogeneity = []
selected_homogeneity = []

prior_flip_rates = []

image_correct = 0
image_total = 0

per_fold_audit = []


# =====================================================================
# 7. AUDIT EACH SAVED OUTER FOLD
# =====================================================================

for _, fr in fold_results.iterrows():

    fold = int(
        fr["fold"]
    )

    print("\n" + "=" * 90)
    print(f"AUDITING OUTER FOLD {fold}")
    print("=" * 90)

    # -----------------------------------------------------------------
    # Reconstruct THIS fold only
    # -----------------------------------------------------------------

    mf = make_fold_manifest(
        fold
    )

    te = (
        mf[
            mf["split"] == "test"
        ]
        .reset_index(
            drop=True
        )
    )

    print(
        f"Held-out images: {len(te)} | "
        f"held-out groups: {te['group_key'].nunique()}"
    )

    # -----------------------------------------------------------------
    # Test dataset only.
    # No train loader / sampler needed.
    # -----------------------------------------------------------------

    test_ds = EBHIWeakOrdinalDataset(
        te,
        train=False,
        paired_stain=False,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    # -----------------------------------------------------------------
    # Load saved best checkpoint
    # -----------------------------------------------------------------

    ckpt_path = (
        CHECKPOINT_ROOT
        / f"fold_{fold}_best.pt"
    )

    if not ckpt_path.exists():

        raise FileNotFoundError(
            ckpt_path
        )

    ckpt = torch.load(
        ckpt_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model = (
        MASOR21Net()
        .to(DEVICE)
    )

    model.load_state_dict(
        ckpt["model"]
    )

    model.eval()

    # -----------------------------------------------------------------
    # Fold-specific validation-selected calibration
    # -----------------------------------------------------------------

    threshold = float(
        fr["bin_threshold"]
    )

    selected_lambda = float(
        fr["prior_lambda"]
    )

    print(
        f"Checkpoint epoch: {ckpt['epoch']}"
    )

    print(
        f"Binary threshold: {threshold:.2f}"
    )

    print(
        f"Selected prior λ: {selected_lambda:.2f}"
    )

    # -----------------------------------------------------------------
    # Collect test logits ONCE.
    #
    # Everything below reuses these cached logits.
    # -----------------------------------------------------------------

    cached = collect_logits(
        model,
        test_loader,
        tta=TTA_ENABLE,
    )

    fold_image_correct = 0
    fold_image_total = 0

    fold_flip_rates = []

    # =================================================================
    # Iterate through cached test batches
    # =================================================================

    for chunk in cached:

        b = chunk[
            "bin_logits"
        ]

        g = chunk[
            "grade_logits"
        ]

        masks = (
            chunk["mask"][:, 0]
            .long()
        )

        labels = (
            chunk["grade"]
            .long()
        )

        # --------------------------------------------------------------
        # Ground truth weak semantic target
        #
        # 0 = background
        # 1 = normal
        # 2 = LGD
        # 3 = HGD
        # 4 = ADC
        # --------------------------------------------------------------

        gt_sem = (
            masks
            * (
                labels[:, None, None]
                + 1
            )
        )

        assert gt_sem.ndim == 3

        # ==============================================================
        # A. TEST EVERY λ
        # ==============================================================

        sem_by_lambda = {}

        for lam in LAMBDAS_TO_AUDIT:

            sem, _ = semantic_from_logits(
                b,
                g,
                masks.shape[-2:],
                threshold,
                lam,
            )

            # semantic_from_logits should return B x H x W
            assert sem.ndim == 3, (
                f"Unexpected semantic shape "
                f"for λ={lam}: {sem.shape}"
            )

            assert sem.shape == masks.shape, (
                f"Semantic/mask mismatch: "
                f"{sem.shape} vs {masks.shape}"
            )

            sem_by_lambda[
                lam
            ] = sem

            add_cm(
                global_cm[lam],
                gt_sem,
                sem,
                K,
            )

        # ==============================================================
        # B. IMAGE-LEVEL POOLED GRADE
        # ==============================================================

        # Binary logits are B x 1 x h x w.
        bin_small = (
            torch.sigmoid(b)
            >= threshold
        ).float()

        assert bin_small.ndim == 4
        assert bin_small.shape[1] == 1

        image_logits = masked_lse_pool(
            g,
            bin_small,
            LSE_TAU,
        )

        # B x 4
        assert image_logits.ndim == 2
        assert image_logits.shape[1] == len(
            GRADE_NAMES
        )

        image_pred = (
            image_logits.argmax(
                dim=1
            )
        )

        correct = int(
            (
                image_pred
                == labels
            ).sum()
        )

        n_batch = int(
            len(labels)
        )

        image_correct += correct
        image_total += n_batch

        fold_image_correct += correct
        fold_image_total += n_batch

        # ==============================================================
        # C. STUPID IMAGE-GRADE + PAINT FOREGROUND BASELINE
        #
        # This is the fixed section.
        #
        # We take EXACTLY the foreground from λ=0 MASOR prediction
        # and replace its local grade predictions with ONE image grade.
        # ==============================================================

        sem_raw = (
            sem_by_lambda[0.00]
        )

        # Shape is now guaranteed B x H x W.
        pred_fg = (
            sem_raw > 0
        )

        assert pred_fg.ndim == 3

        assert (
            pred_fg.shape
            == masks.shape
        ), (
            f"pred_fg={pred_fg.shape}, "
            f"masks={masks.shape}"
        )

        # Expand one image diagnosis to every HxW pixel.
        paint_grade = (
            image_pred[
                :,
                None,
                None,
            ]
            + 1
        ).expand_as(
            masks
        )

        assert (
            paint_grade.shape
            == masks.shape
        )

        paint = torch.where(
            pred_fg,
            paint_grade,
            torch.zeros_like(
                masks
            ),
        )

        assert (
            paint.shape
            == masks.shape
        )

        add_cm(
            paint_cm,
            gt_sem,
            paint,
            K,
        )

        # ==============================================================
        # D. ORACLE TRUE-GRADE PAINT
        #
        # Same predicted foreground, but uses TRUE image grade.
        #
        # THIS IS NOT MODEL PERFORMANCE.
        # It is a benchmark-shortcut upper-bound control.
        # ==============================================================

        oracle_grade = (
            labels[
                :,
                None,
                None,
            ]
            + 1
        ).expand_as(
            masks
        )

        assert (
            oracle_grade.shape
            == masks.shape
        )

        oracle_paint = torch.where(
            pred_fg,
            oracle_grade,
            torch.zeros_like(
                masks
            ),
        )

        assert (
            oracle_paint.shape
            == masks.shape
        )

        add_cm(
            oracle_paint_cm,
            gt_sem,
            oracle_paint,
            K,
        )

        # ==============================================================
        # E. SELECTED-λ MAP
        # ==============================================================

        if selected_lambda in sem_by_lambda:

            sem_selected = (
                sem_by_lambda[
                    selected_lambda
                ]
            )

        else:

            sem_selected, _ = semantic_from_logits(
                b,
                g,
                masks.shape[-2:],
                threshold,
                selected_lambda,
            )

        assert (
            sem_selected.shape
            == sem_raw.shape
        )

        # ==============================================================
        # F. PER-IMAGE HOMOGENEITY + PRIOR FLIP RATE
        # ==============================================================

        for i in range(
            len(labels)
        ):

            raw_h = image_homogeneity(
                sem_raw[i]
            )

            selected_h = image_homogeneity(
                sem_selected[i]
            )

            raw_homogeneity.append(
                raw_h
            )

            selected_homogeneity.append(
                selected_h
            )

            fg_i = pred_fg[i]

            if fg_i.any():

                # What fraction of foreground pixels change
                # grade when the image prior is introduced?
                changed = (
                    sem_raw[i][fg_i]
                    != sem_selected[i][fg_i]
                )

                flip_rate = float(
                    changed
                    .float()
                    .mean()
                    .item()
                )

                prior_flip_rates.append(
                    flip_rate
                )

                fold_flip_rates.append(
                    flip_rate
                )

    # -----------------------------------------------------------------
    # Fold summary
    # -----------------------------------------------------------------

    fold_acc = (
        fold_image_correct
        / max(
            fold_image_total,
            1,
        )
    )

    fold_mean_flip = (
        float(
            np.mean(
                fold_flip_rates
            )
        )
        if len(
            fold_flip_rates
        )
        else np.nan
    )

    per_fold_audit.append(
        {
            "fold": fold,
            "image_grade_accuracy": fold_acc,
            "selected_lambda": selected_lambda,
            "mean_prior_pixel_flip_rate": fold_mean_flip,
        }
    )

    print(
        f"Image-grade accuracy: "
        f"{fold_acc:.4f}"
    )

    print(
        f"Mean foreground pixels changed by prior: "
        f"{fold_mean_flip:.4f}"
    )

    # Free GPU memory before next fold
    del model
    del cached

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =====================================================================
# 8. COMPUTE POOLED OOF METRICS
# =====================================================================

print("\n" + "=" * 100)
print("POOLED OOF PRIOR-SENSITIVITY AUDIT")
print("=" * 100)

audit_rows = []

for lam in LAMBDAS_TO_AUDIT:

    m = metrics_from_cm(
        global_cm[lam]
    )

    audit_rows.append(
        {
            "mode": (
                f"MASOR λ={lam:.2f}"
            ),
            **m,
        }
    )


# ---------------------------------------------------------------------
# Image-grade + paint baseline
# ---------------------------------------------------------------------

paint_metrics = metrics_from_cm(
    paint_cm
)

audit_rows.append(
    {
        "mode": (
            "IMAGE-GRADE + PAINT FOREGROUND"
        ),
        **paint_metrics,
    }
)


# ---------------------------------------------------------------------
# Oracle paint control
# ---------------------------------------------------------------------

oracle_metrics = metrics_from_cm(
    oracle_paint_cm
)

audit_rows.append(
    {
        "mode": (
            "ORACLE TRUE-GRADE PAINT "
            "(NOT MODEL)"
        ),
        **oracle_metrics,
    }
)


audit_df = pd.DataFrame(
    audit_rows
)


# =====================================================================
# 9. DISPLAY MAIN RESULT TABLE
# =====================================================================

main_cols = [
    "mode",
    "macro_fg_dice",
    "gmean_fg_dice",
    "dice_normal_gland",
    "dice_low_grade",
    "dice_high_grade",
    "dice_adk_invasion",
]

display(
    audit_df[
        main_cols
    ].style.format(
        {
            col: "{:.4f}"
            for col in main_cols
            if col != "mode"
        }
    )
)


# =====================================================================
# 10. IMAGE-LEVEL GRADE ACCURACY
# =====================================================================

image_grade_accuracy = (
    image_correct
    / max(
        image_total,
        1,
    )
)

print("\n" + "=" * 100)
print("IMAGE-LEVEL DIAGNOSIS")
print("=" * 100)

print(
    f"Pooled image-level grade accuracy: "
    f"{image_grade_accuracy:.4f}"
)

print(
    f"Correct images: "
    f"{image_correct}/{image_total}"
)


# =====================================================================
# 11. LOCAL MAP HOMOGENEITY
# =====================================================================

raw_h = np.asarray(
    raw_homogeneity,
    dtype=float,
)

selected_h = np.asarray(
    selected_homogeneity,
    dtype=float,
)

flips = np.asarray(
    prior_flip_rates,
    dtype=float,
)


print("\n" + "=" * 100)
print("LOCAL MAP HOMOGENEITY")
print("=" * 100)


print(
    "λ=0 mean dominant-grade fraction: "
    f"{np.nanmean(raw_h):.4f}"
)

print(
    "selected-λ mean dominant-grade fraction: "
    f"{np.nanmean(selected_h):.4f}"
)


print(
    "\nλ=0 median dominant-grade fraction: "
    f"{np.nanmedian(raw_h):.4f}"
)

print(
    "selected-λ median dominant-grade fraction: "
    f"{np.nanmedian(selected_h):.4f}"
)


print(
    "\nλ=0 images with >=99% one grade: "
    f"{np.nanmean(raw_h >= 0.99):.4f}"
)

print(
    "selected-λ images with >=99% one grade: "
    f"{np.nanmean(selected_h >= 0.99):.4f}"
)


# =====================================================================
# 12. PRIOR EFFECT
# =====================================================================

print("\n" + "=" * 100)
print("IMAGE-PRIOR EFFECT")
print("=" * 100)

print(
    "Mean fraction of foreground pixels whose grade "
    "changes due to the selected image prior:"
)

print(
    f"  mean   = "
    f"{np.nanmean(flips):.4f}"
)

print(
    f"  median = "
    f"{np.nanmedian(flips):.4f}"
)

print(
    f"  p90    = "
    f"{np.nanquantile(flips, 0.90):.4f}"
)

print(
    f"  max    = "
    f"{np.nanmax(flips):.4f}"
)


# =====================================================================
# 13. PER-FOLD DIAGNOSTIC TABLE
# =====================================================================

print("\n" + "=" * 100)
print("PER-FOLD PRIOR AUDIT")
print("=" * 100)

per_fold_audit_df = pd.DataFrame(
    per_fold_audit
)

display(
    per_fold_audit_df.style.format(
        {
            "image_grade_accuracy": "{:.4f}",
            "selected_lambda": "{:.2f}",
            "mean_prior_pixel_flip_rate": "{:.4f}",
        }
    )
)


# =====================================================================
# 14. SAVE AUDIT OUTPUTS
# =====================================================================

audit_csv = (
    ARTIFACT_ROOT
    / "MASOR21_prior_sensitivity_audit.csv"
)

fold_audit_csv = (
    ARTIFACT_ROOT
    / "MASOR21_prior_effect_by_fold.csv"
)

audit_df.to_csv(
    audit_csv,
    index=False,
)

per_fold_audit_df.to_csv(
    fold_audit_csv,
    index=False,
)


print("\nSaved:")
print(" -", audit_csv)
print(" -", fold_audit_csv)


# =====================================================================
# 15. INTERPRETATION GUIDE
# =====================================================================

print("\n" + "=" * 100)
print("HOW TO READ THIS")
print("=" * 100)

print(
"""
GREEN FLAG:
-----------
MASOR λ=0 is already strong,
AND image-grade + paint is clearly worse,
AND selected λ only changes a modest number of pixels.

=> Local prototype morphology is genuinely contributing.


YELLOW FLAG:
------------
λ=0 is decent but substantially below λ=.75,
and the image-prior changes many pixels.

=> Local morphology works, but image-level evidence is doing
   significant correction.


RED FLAG:
---------
IMAGE-GRADE + PAINT FOREGROUND nearly matches MASOR λ=.75,
while λ=0 is much worse.

=> EBHI semantic Dice is primarily rewarding:
      binary segmentation
      + image classification
      + painting the whole foreground.

   In that case, do NOT interpret the ~0.80 semantic Dice as
   independently validated fine-grained pixel grading.


ORACLE CONTROL:
---------------
ORACLE TRUE-GRADE PAINT should be very high.

That row intentionally cheats.

It measures how high the fabricated EBHI dense target can score
if image diagnosis is known perfectly and localisation is good.


IMPORTANT:
----------
λ=1.0 and λ=1.5 are only post-hoc sensitivity probes.

Do NOT select a new λ using these outer-test results.
"""
)

print("=" * 100)
print("AUDIT COMPLETE")
print("=" * 100)

In [ ]:
# =====================================================================
# MASOR-2.1 FORENSIC AUDIT C
# MORPHOLOGY / COLOR / CONTEXT SHORTCUT AUDIT
# =====================================================================
#
# QUESTION:
# What information is the grade branch actually using?
#
# Held-out test images are transformed into:
#
#   ORIGINAL
#   FOREGROUND ONLY      -> removes surrounding context
#   GRAYSCALE            -> removes H&E colour information
#   LOWPASS              -> destroys fine cellular morphology
#   BLOCK SHUFFLE        -> destroys global architecture
#   TISSUE MEAN COLOR    -> destroys morphology, keeps mean stain colour
#   BACKGROUND ONLY      -> removes annotated foreground entirely
#
# Grade prediction is pooled inside the TRUE EBHI foreground mask.
# This is intentional:
#
#   We want to isolate the GRADE REPRESENTATION from localisation.
#
# Therefore these are diagnostic counterfactuals, NOT clinical
# performance numbers.
#
# NO TRAINING.
# NO HYPERPARAMETER SELECTION.
# =====================================================================

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
)

from IPython.display import display


# =====================================================================
# 0. RECOVER EXPERIMENT PATHS
# =====================================================================

if "WORK_ROOT" not in globals():

    hits = list(
        Path("/kaggle/working").rglob("fold_0_best.pt")
    )

    if len(hits) == 0:
        raise FileNotFoundError(
            "Could not locate fold_0_best.pt under /kaggle/working."
        )

    preferred = [
        p for p in hits
        if "masor21" in str(p).lower()
    ]

    ckpt0 = (
        preferred[0]
        if preferred
        else hits[0]
    )

    CHECKPOINT_ROOT = ckpt0.parent
    WORK_ROOT = CHECKPOINT_ROOT.parent
    ARTIFACT_ROOT = WORK_ROOT / "artifacts"

else:

    WORK_ROOT = Path(WORK_ROOT)

    ARTIFACT_ROOT = Path(
        globals().get(
            "ARTIFACT_ROOT",
            WORK_ROOT / "artifacts",
        )
    )

    CHECKPOINT_ROOT = Path(
        globals().get(
            "CHECKPOINT_ROOT",
            WORK_ROOT / "checkpoints",
        )
    )


print("=" * 100)
print("MASOR-2.1 FORENSIC AUDIT C")
print("=" * 100)

print("Experiment:", WORK_ROOT)


# =====================================================================
# 1. LOAD EXPERIMENT
# =====================================================================

master = pd.read_csv(
    WORK_ROOT / "master_manifest.csv"
)

fold_results = pd.read_csv(
    ARTIFACT_ROOT / "fold_comparison.csv"
)


# =====================================================================
# 2. REQUIRED DEFINITIONS
# =====================================================================

required = [
    "MASOR21Net",
    "EBHIWeakOrdinalDataset",
    "make_fold_manifest",
    "masked_lse_pool",
    "GRADE_NAMES",
    "IMAGE_SIZE",
    "DEVICE",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:

    raise RuntimeError(
        "Missing notebook definitions:\n\n"
        + "\n".join(
            f" - {x}"
            for x in missing
        )
        + "\n\nRerun the definition cells only. "
          "Do NOT retrain."
    )


# Safe defaults if kernel forgot these
BATCH_SIZE_AUDIT = int(
    globals().get(
        "BATCH_SIZE",
        8,
    )
)

NUM_WORKERS_AUDIT = int(
    globals().get(
        "NUM_WORKERS",
        2,
    )
)

PIN_MEMORY_AUDIT = (
    DEVICE.type == "cuda"
)


# =====================================================================
# 3. EXACT NORMALISATION USED BY MASOR
# =====================================================================

AUDIT_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32,
).view(
    1, 3, 1, 1
)

AUDIT_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32,
).view(
    1, 3, 1, 1
)


def denormalise_to_01(x):

    """
    x:
        normalised Bx3xHxW MASOR tensor

    returns:
        RGB in [0,1]
    """

    mean = AUDIT_MEAN.to(
        x.device
    )

    std = AUDIT_STD.to(
        x.device
    )

    rgb = (
        x * std
        + mean
    )

    return rgb.clamp(
        0.0,
        1.0,
    )


def normalise_from_01(rgb):

    mean = AUDIT_MEAN.to(
        rgb.device
    )

    std = AUDIT_STD.to(
        rgb.device
    )

    return (
        rgb - mean
    ) / std


# =====================================================================
# 4. COUNTERFACTUAL TRANSFORMS
# =====================================================================

def make_grayscale(rgb):

    """
    Preserve morphology/luminance.
    Destroy RGB stain colour information.
    """

    gray = (
        0.2989 * rgb[:, 0:1]
        + 0.5870 * rgb[:, 1:2]
        + 0.1140 * rgb[:, 2:3]
    )

    return gray.repeat(
        1,
        3,
        1,
        1,
    )


def make_foreground_only(rgb, gt_mask):

    """
    Preserve real annotated foreground.
    Replace everything outside it with white.
    """

    mask3 = gt_mask.repeat(
        1,
        3,
        1,
        1,
    ).bool()

    white = torch.ones_like(
        rgb
    )

    return torch.where(
        mask3,
        rgb,
        white,
    )


def make_background_only(rgb, gt_mask):

    """
    Remove annotated foreground.
    Preserve surrounding histology.
    """

    mask3 = gt_mask.repeat(
        1,
        3,
        1,
        1,
    ).bool()

    white = torch.ones_like(
        rgb
    )

    return torch.where(
        mask3,
        white,
        rgb,
    )


def make_lowpass(rgb):

    """
    Aggressively destroys nuclei/gland microdetail.

    224 -> 28 -> 224

    Coarse colour + coarse architecture remain.
    """

    tiny = F.interpolate(
        rgb,
        size=(28, 28),
        mode="bilinear",
        align_corners=False,
    )

    return F.interpolate(
        tiny,
        size=rgb.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )


def make_block_shuffle(
    rgb,
    block_size=32,
    seed=42,
):

    """
    Preserve local texture + colour.
    Destroy global gland arrangement.

    IMAGE_SIZE=224 is divisible by 32:
        7 x 7 blocks.
    """

    B, C, H, W = rgb.shape

    assert H % block_size == 0
    assert W % block_size == 0

    gh = H // block_size
    gw = W // block_size

    out = torch.empty_like(
        rgb
    )

    generator = torch.Generator(
        device="cpu"
    )

    generator.manual_seed(
        seed
    )

    for b in range(B):

        blocks = []

        for iy in range(gh):
            for ix in range(gw):

                block = rgb[
                    b,
                    :,
                    iy * block_size:
                    (iy + 1) * block_size,
                    ix * block_size:
                    (ix + 1) * block_size,
                ]

                blocks.append(
                    block.clone()
                )

        perm = torch.randperm(
            len(blocks),
            generator=generator,
        ).tolist()

        k = 0

        for iy in range(gh):
            for ix in range(gw):

                out[
                    b,
                    :,
                    iy * block_size:
                    (iy + 1) * block_size,
                    ix * block_size:
                    (ix + 1) * block_size,
                ] = blocks[
                    perm[k]
                ]

                k += 1

    return out


def make_tissue_mean_color(
    rgb,
    gt_mask,
):

    """
    THE PURE COLOUR SHORTCUT CONTROL.

    All morphological structure is removed.

    Every foreground pixel becomes the image's
    mean foreground RGB colour.

    Background becomes white.
    """

    B, C, H, W = rgb.shape

    out = torch.ones_like(
        rgb
    )

    for b in range(B):

        m = gt_mask[
            b,
            0
        ].bool()

        if m.any():

            pixels = rgb[
                b,
                :,
                m,
            ]

            mean_rgb = pixels.mean(
                dim=1
            )

            for c in range(C):

                out[
                    b,
                    c,
                ][m] = mean_rgb[c]

    return out


# =====================================================================
# 5. MODEL-GRADE READOUT
# =====================================================================

@torch.no_grad()
def grade_from_image(
    model,
    x,
    gt_mask,
):

    """
    Run MASOR and obtain image grade.

    IMPORTANT:
    Uses TRUE EBHI mask only as the pooling support.

    This deliberately isolates grade recognition from
    foreground localisation.
    """

    out = model(
        x
    )

    grade_logits = out[
        "grade_logits"
    ]

    # Resize true binary mask to feature resolution
    mask_small = F.interpolate(
        gt_mask.float(),
        size=grade_logits.shape[-2:],
        mode="nearest",
    )

    pooled = masked_lse_pool(
        grade_logits,
        mask_small,
        LSE_TAU,
    )

    probs = F.softmax(
        pooled,
        dim=1,
    )

    pred = probs.argmax(
        dim=1
    )

    confidence = probs.max(
        dim=1
    ).values

    expected_grade = (
        probs
        * torch.arange(
            len(GRADE_NAMES),
            device=probs.device,
            dtype=probs.dtype,
        )[None]
    ).sum(
        dim=1
    )

    entropy = -(
        probs
        * torch.log(
            probs.clamp_min(
                1e-8
            )
        )
    ).sum(
        dim=1
    )

    entropy = (
        entropy
        / np.log(
            len(GRADE_NAMES)
        )
    )

    return {
        "pred": pred,
        "probs": probs,
        "confidence": confidence,
        "expected_grade": expected_grade,
        "entropy": entropy,
    }


# =====================================================================
# 6. VARIANTS TO TEST
# =====================================================================

VARIANT_NAMES = [
    "original",
    "foreground_only",
    "grayscale",
    "lowpass_28",
    "block_shuffle_32",
    "tissue_mean_color",
    "background_only",
]


# =====================================================================
# 7. ACCUMULATORS
# =====================================================================

audit_rows = []

per_image_rows = []

example_bank = {
    grade: None
    for grade in GRADE_NAMES
}


# =====================================================================
# 8. OUTER-FOLD AUDIT
# =====================================================================

for _, fr in fold_results.iterrows():

    fold = int(
        fr["fold"]
    )

    print("\n" + "=" * 100)
    print(
        f"COUNTERFACTUAL AUDIT — OUTER FOLD {fold}"
    )
    print("=" * 100)

    mf = make_fold_manifest(
        fold
    )

    te = (
        mf[
            mf["split"]
            == "test"
        ]
        .reset_index(
            drop=True
        )
    )

    ds = EBHIWeakOrdinalDataset(
        te,
        train=False,
        paired_stain=False,
    )

    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE_AUDIT,
        shuffle=False,
        num_workers=NUM_WORKERS_AUDIT,
        pin_memory=PIN_MEMORY_AUDIT,
    )

    ckpt_path = (
        CHECKPOINT_ROOT
        / f"fold_{fold}_best.pt"
    )

    ckpt = torch.load(
        ckpt_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model = MASOR21Net().to(
        DEVICE
    )

    model.load_state_dict(
        ckpt["model"]
    )

    model.eval()

    fold_correct = {
        v: 0
        for v in VARIANT_NAMES
    }

    fold_total = 0

    for batch_i, batch in enumerate(
        loader
    ):

        clean = (
            batch["clean"]
            .to(
                DEVICE,
                non_blocking=True,
            )
        )

        gt_mask = (
            batch["mask"]
            .to(
                DEVICE,
                non_blocking=True,
            )
        )

        labels = (
            batch["grade"]
            .to(
                DEVICE,
                non_blocking=True,
            )
        )

        # -------------------------------------------------------------
        # Recover RGB [0,1]
        # -------------------------------------------------------------

        rgb = denormalise_to_01(
            clean
        )

        # -------------------------------------------------------------
        # Construct all counterfactuals
        # -------------------------------------------------------------

        variants_rgb = {
            "original":
                rgb,

            "foreground_only":
                make_foreground_only(
                    rgb,
                    gt_mask,
                ),

            "grayscale":
                make_grayscale(
                    rgb
                ),

            "lowpass_28":
                make_lowpass(
                    rgb
                ),

            "block_shuffle_32":
                make_block_shuffle(
                    rgb,
                    block_size=32,
                    seed=(
                        10000
                        + fold * 100
                        + batch_i
                    ),
                ),

            "tissue_mean_color":
                make_tissue_mean_color(
                    rgb,
                    gt_mask,
                ),

            "background_only":
                make_background_only(
                    rgb,
                    gt_mask,
                ),
        }

        # -------------------------------------------------------------
        # Evaluate every variant
        # -------------------------------------------------------------

        variant_outputs = {}

        for variant_name, variant_rgb in variants_rgb.items():

            x_variant = normalise_from_01(
                variant_rgb
            )

            out = grade_from_image(
                model,
                x_variant,
                gt_mask,
            )

            variant_outputs[
                variant_name
            ] = out

            pred = out[
                "pred"
            ]

            fold_correct[
                variant_name
            ] += int(
                (
                    pred
                    == labels
                ).sum()
            )

        fold_total += len(
            labels
        )

        # -------------------------------------------------------------
        # Per-image records
        # -------------------------------------------------------------

        for i in range(
            len(labels)
        ):

            true_idx = int(
                labels[i].item()
            )

            true_name = GRADE_NAMES[
                true_idx
            ]

            record = {
                "fold":
                    fold,

                "sample_id":
                    batch["sample_id"][i],

                "group_key":
                    batch["group_key"][i],

                "true_grade":
                    true_name,

                "true_grade_idx":
                    true_idx,
            }

            original_pred = int(
                variant_outputs[
                    "original"
                ]["pred"][i].item()
            )

            for v in VARIANT_NAMES:

                o = variant_outputs[v]

                pred_idx = int(
                    o["pred"][i].item()
                )

                record[
                    f"{v}_pred"
                ] = pred_idx

                record[
                    f"{v}_correct"
                ] = int(
                    pred_idx
                    == true_idx
                )

                record[
                    f"{v}_confidence"
                ] = float(
                    o["confidence"][i]
                    .item()
                )

                record[
                    f"{v}_expected_grade"
                ] = float(
                    o["expected_grade"][i]
                    .item()
                )

                record[
                    f"{v}_entropy"
                ] = float(
                    o["entropy"][i]
                    .item()
                )

                record[
                    f"{v}_agrees_original"
                ] = int(
                    pred_idx
                    == original_pred
                )

            per_image_rows.append(
                record
            )

            # ---------------------------------------------------------
            # Save one random visual example for each class
            # ---------------------------------------------------------

            if example_bank[
                true_name
            ] is None:

                example_bank[
                    true_name
                ] = {
                    name:
                    tensor[
                        i
                    ].detach()
                    .cpu()
                    .permute(
                        1,
                        2,
                        0,
                    )
                    .numpy()
                    for name, tensor
                    in variants_rgb.items()
                }

    # -------------------------------------------------------------
    # Fold summary
    # -------------------------------------------------------------

    for v in VARIANT_NAMES:

        acc = (
            fold_correct[v]
            / max(
                fold_total,
                1,
            )
        )

        print(
            f"{v:22s}: "
            f"{acc:.4f}"
        )

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =====================================================================
# 9. CREATE OOF TABLE
# =====================================================================

per_image_df = pd.DataFrame(
    per_image_rows
)

print("\n" + "=" * 100)
print("POOLED OOF COUNTERFACTUAL RESULTS")
print("=" * 100)


# =====================================================================
# 10. METRICS PER VARIANT
# =====================================================================

summary_rows = []

y_true = per_image_df[
    "true_grade_idx"
].to_numpy()

for v in VARIANT_NAMES:

    y_pred = per_image_df[
        f"{v}_pred"
    ].to_numpy()

    confidence = per_image_df[
        f"{v}_confidence"
    ].to_numpy()

    entropy = per_image_df[
        f"{v}_entropy"
    ].to_numpy()

    expected = per_image_df[
        f"{v}_expected_grade"
    ].to_numpy()

    agreement = per_image_df[
        f"{v}_agrees_original"
    ].to_numpy()

    acc = accuracy_score(
        y_true,
        y_pred,
    )

    bacc = balanced_accuracy_score(
        y_true,
        y_pred,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    qwk = cohen_kappa_score(
        y_true,
        y_pred,
        weights="quadratic",
    )

    mae_hard = np.mean(
        np.abs(
            y_true
            - y_pred
        )
    )

    mae_expected = np.mean(
        np.abs(
            y_true
            - expected
        )
    )

    far_error = np.mean(
        np.abs(
            y_true
            - y_pred
        ) >= 2
    )

    summary_rows.append({
        "variant":
            v,

        "accuracy":
            acc,

        "balanced_accuracy":
            bacc,

        "macro_F1":
            macro_f1,

        "QWK":
            qwk,

        "hard_MAE":
            mae_hard,

        "expected_grade_MAE":
            mae_expected,

        "far_error_rate":
            far_error,

        "mean_confidence":
            np.mean(
                confidence
            ),

        "mean_entropy":
            np.mean(
                entropy
            ),

        "agreement_with_original":
            np.mean(
                agreement
            ),
    })


summary_df = pd.DataFrame(
    summary_rows
)

display(
    summary_df.style.format({
        "accuracy":
            "{:.4f}",

        "balanced_accuracy":
            "{:.4f}",

        "macro_F1":
            "{:.4f}",

        "QWK":
            "{:.4f}",

        "hard_MAE":
            "{:.4f}",

        "expected_grade_MAE":
            "{:.4f}",

        "far_error_rate":
            "{:.4f}",

        "mean_confidence":
            "{:.4f}",

        "mean_entropy":
            "{:.4f}",

        "agreement_with_original":
            "{:.4f}",
    })
)


# =====================================================================
# 11. NORMALISE PERFORMANCE RELATIVE TO ORIGINAL
# =====================================================================

original_acc = float(
    summary_df.loc[
        summary_df[
            "variant"
        ]
        == "original",
        "accuracy",
    ].iloc[0]
)

original_qwk = float(
    summary_df.loc[
        summary_df[
            "variant"
        ]
        == "original",
        "QWK",
    ].iloc[0]
)


relative_rows = []

for _, r in summary_df.iterrows():

    relative_rows.append({
        "variant":
            r["variant"],

        "accuracy":
            r["accuracy"],

        "accuracy_retained_%":
            100
            * r["accuracy"]
            / max(
                original_acc,
                1e-12,
            ),

        "QWK":
            r["QWK"],

        "QWK_retained_%":
            (
                100
                * r["QWK"]
                / original_qwk
                if abs(
                    original_qwk
                ) > 1e-12
                else np.nan
            ),
    })


relative_df = pd.DataFrame(
    relative_rows
)

print("\nPERFORMANCE RETAINED RELATIVE TO CLEAN IMAGE")

display(
    relative_df.style.format({
        "accuracy":
            "{:.4f}",

        "accuracy_retained_%":
            "{:.1f}%",

        "QWK":
            "{:.4f}",

        "QWK_retained_%":
            "{:.1f}%",
    })
)


# =====================================================================
# 12. CONFUSION MATRICES
# =====================================================================

fig, axes = plt.subplots(
    2,
    4,
    figsize=(18, 9),
)

axes = axes.ravel()

for ax_i, v in enumerate(
    VARIANT_NAMES
):

    y_pred = per_image_df[
        f"{v}_pred"
    ].to_numpy()

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(
            len(GRADE_NAMES)
        ),
    )

    # Row normalise
    cm_norm = (
        cm
        / np.maximum(
            cm.sum(
                axis=1,
                keepdims=True,
            ),
            1,
        )
    )

    ax = axes[
        ax_i
    ]

    im = ax.imshow(
        cm_norm,
        vmin=0,
        vmax=1,
        cmap="Blues",
    )

    ax.set_title(
        v
    )

    ax.set_xticks(
        range(
            len(GRADE_NAMES)
        )
    )

    ax.set_yticks(
        range(
            len(GRADE_NAMES)
        )
    )

    ax.set_xticklabels(
        [
            "N",
            "LGD",
            "HGD",
            "ADC",
        ],
        rotation=45,
    )

    ax.set_yticklabels(
        [
            "N",
            "LGD",
            "HGD",
            "ADC",
        ]
    )

    for i in range(
        len(GRADE_NAMES)
    ):
        for j in range(
            len(GRADE_NAMES)
        ):

            ax.text(
                j,
                i,
                f"{cm_norm[i,j]:.2f}",
                ha="center",
                va="center",
                fontsize=8,
            )

    ax.set_xlabel(
        "Predicted"
    )

    ax.set_ylabel(
        "True"
    )


# Hide unused panel
for j in range(
    len(VARIANT_NAMES),
    len(axes),
):
    axes[j].axis(
        "off"
    )


fig.suptitle(
    "MASOR-2.1 counterfactual grade confusion",
    fontsize=16,
    fontweight="bold",
)

plt.tight_layout()
plt.show()


# =====================================================================
# 13. VISUALISE WHAT THE COUNTERFACTUALS ACTUALLY LOOK LIKE
# =====================================================================

fig, axes = plt.subplots(
    len(GRADE_NAMES),
    len(VARIANT_NAMES),
    figsize=(21, 12),
)

for r, grade in enumerate(
    GRADE_NAMES
):

    ex = example_bank[
        grade
    ]

    for c, v in enumerate(
        VARIANT_NAMES
    ):

        ax = axes[
            r,
            c,
        ]

        ax.imshow(
            np.clip(
                ex[v],
                0,
                1,
            )
        )

        if r == 0:
            ax.set_title(
                v,
                fontsize=10,
                fontweight="bold",
            )

        if c == 0:
            ax.set_ylabel(
                grade,
                fontsize=10,
                fontweight="bold",
            )

        ax.set_xticks([])
        ax.set_yticks([])


plt.suptitle(
    "Actual counterfactual inputs used in Audit C",
    fontsize=16,
    fontweight="bold",
)

plt.tight_layout()
plt.show()


# =====================================================================
# 14. SAVE EVERYTHING
# =====================================================================

summary_path = (
    ARTIFACT_ROOT
    / "MASOR21_counterfactual_shortcut_audit.csv"
)

per_image_path = (
    ARTIFACT_ROOT
    / "MASOR21_counterfactual_per_image.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

per_image_df.to_csv(
    per_image_path,
    index=False,
)

print("\nSaved:")
print(" -", summary_path)
print(" -", per_image_path)


# =====================================================================
# 15. AUTOMATIC SANITY FLAGS
# =====================================================================

def get_acc(name):

    return float(
        summary_df.loc[
            summary_df[
                "variant"
            ] == name,
            "accuracy",
        ].iloc[0]
    )


acc_original = get_acc(
    "original"
)

acc_fg = get_acc(
    "foreground_only"
)

acc_gray = get_acc(
    "grayscale"
)

acc_blur = get_acc(
    "lowpass_28"
)

acc_shuffle = get_acc(
    "block_shuffle_32"
)

acc_color = get_acc(
    "tissue_mean_color"
)

acc_bg = get_acc(
    "background_only"
)


print("\n" + "=" * 100)
print("AUTOMATIC INTERPRETATION")
print("=" * 100)

print(
    f"""
Original accuracy:               {acc_original:.3f}

Foreground-only accuracy:        {acc_fg:.3f}
Grayscale accuracy:              {acc_gray:.3f}
Low-pass accuracy:               {acc_blur:.3f}
Block-shuffle accuracy:          {acc_shuffle:.3f}
Tissue-mean-colour accuracy:     {acc_color:.3f}
Background-only accuracy:        {acc_bg:.3f}

Four-class chance accuracy:      0.250
"""
)


# Pure average-colour shortcut
if acc_color >= 0.50:

    print(
        "🚨 STRONG WARNING: mean tissue colour alone "
        "predicts grade far above chance."
    )

elif acc_color >= 0.35:

    print(
        "⚠️ Mean tissue colour contains noticeable class information."
    )

else:

    print(
        "✓ Mean tissue colour alone is weak."
    )


# Background/context
if acc_bg >= 0.50:

    print(
        "🚨 STRONG WARNING: background/context alone "
        "predicts grade very well."
    )

elif acc_bg >= 0.35:

    print(
        "⚠️ Background/context carries non-trivial grade information."
    )

else:

    print(
        "✓ Background-only performance is close to weak/chance levels."
    )


# Fine morphology
if acc_blur >= 0.80 * acc_original:

    print(
        "⚠️ Most diagnostic performance survives severe low-pass filtering. "
        "Fine morphology may not be essential to the model."
    )

else:

    print(
        "✓ Removing fine morphology causes a substantial performance drop."
    )


# Global architecture
if acc_shuffle >= 0.90 * acc_original:

    print(
        "⚠️ Block shuffling barely hurts performance. "
        "The model may rely mainly on local texture rather than global architecture."
    )

else:

    print(
        "✓ Destroying spatial architecture meaningfully hurts grading."
    )


# Stain dependence
if acc_gray >= 0.90 * acc_original:

    print(
        "✓ Grade recognition largely survives removal of colour."
    )

elif acc_gray < 0.65 * acc_original:

    print(
        "⚠️ Large grayscale collapse: stain colour may be carrying substantial signal."
    )

else:

    print(
        "• Colour contributes, but morphology also appears informative."
    )


# Foreground dependence
if acc_fg >= 0.90 * acc_original:

    print(
        "✓ Foreground tissue alone preserves most diagnostic performance."
    )

else:

    print(
        "⚠️ Removing surrounding context substantially reduces grading performance."
    )


print("\n" + "=" * 100)
print("AUDIT C COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# MASOR-2.1 — RUN ONE RANDOM IMAGE THROUGH THE FROZEN CHECKPOINT
#
# Paste this as a new cell AFTER cells 0-13 have been run
# (so the model class, dataset class, inference helpers,
# master manifest, and fold_comparison.csv are all in memory
# or on disk). Works whether you just finished training or
# came back to the notebook and reran only the definitions.
#
# Usage:
#     run_one_random()                    # random image, any class, any fold
#     run_one_random(cls="high_grade")    # random HG image
#     run_one_random(cls="adk_invasion", seed=42)   # reproducible
#     run_one_random(sample_id="GT2013987-3-400-002.png")  # specific image
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from PIL import Image


# ------------------------------------------------------------
# Recover paths + tables if kernel was restarted
# ------------------------------------------------------------
if "WORK_ROOT" not in globals():
    candidates = list(Path("/kaggle/working").rglob("fold_0_best.pt"))
    if not candidates:
        raise FileNotFoundError("No fold_0_best.pt found under /kaggle/working — retrain or restore checkpoints first.")
    preferred = [p for p in candidates if "masor21" in str(p).lower()]
    ckpt0 = preferred[0] if preferred else candidates[0]
    CHECKPOINT_ROOT = ckpt0.parent
    WORK_ROOT = CHECKPOINT_ROOT.parent
    ARTIFACT_ROOT = WORK_ROOT / "artifacts"
else:
    WORK_ROOT = Path(WORK_ROOT)
    CHECKPOINT_ROOT = WORK_ROOT / "checkpoints"
    ARTIFACT_ROOT = WORK_ROOT / "artifacts"

if "master" not in globals():
    master = pd.read_csv(WORK_ROOT / "master_manifest.csv")
if "fold_results" not in globals():
    fold_results = pd.read_csv(ARTIFACT_ROOT / "fold_comparison.csv")

for name in ["MASOR21Net", "EBHIWeakOrdinalDataset", "infer_logits", "semantic_from_logits"]:
    if name not in globals():
        raise RuntimeError(
            f"'{name}' is not defined. Rerun the definition cells (config, dataset, model, inference helpers) — "
            "no retraining needed."
        )


# ------------------------------------------------------------
# Palette (matches your existing viz cell)
# ------------------------------------------------------------
SEMANTIC_COLORS = ["black", "limegreen", "gold", "darkorange", "red"]
SEMANTIC_LABELS = ["Background", "Normal", "LGD", "HGD", "ADC"]
GRADE_PRETTY = {
    "normal_gland": "Normal gland",
    "low_grade":    "Low-grade IN",
    "high_grade":   "High-grade IN",
    "adk_invasion": "Adenocarcinoma",
}
CMAP = ListedColormap(SEMANTIC_COLORS)


# ------------------------------------------------------------
# Checkpoint cache — avoid reloading the same fold repeatedly
# ------------------------------------------------------------
_MODEL_CACHE = {}

def _load_fold_model(fold: int):
    if fold in _MODEL_CACHE:
        return _MODEL_CACHE[fold]
    ckpt_path = CHECKPOINT_ROOT / f"fold_{fold}_best.pt"
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model = MASOR21Net().to(DEVICE)
    model.load_state_dict(ckpt["model"])
    model.eval()
    _MODEL_CACHE[fold] = model
    return model


# ------------------------------------------------------------
# The one-image runner
# ------------------------------------------------------------
def run_one_random(cls: str | None = None,
                   sample_id: str | None = None,
                   seed: int | None = None,
                   show: bool = True):
    """
    Run ONE image through its held-out fold's model and display it.

    cls        : one of "normal_gland","low_grade","high_grade","adk_invasion"
                 (None = any class)
    sample_id  : force a specific image (overrides cls + seed)
    seed       : reproducible pick
    show       : draw the matplotlib figure

    Returns dict with the arrays + metrics so you can inspect programmatically.
    """
    rng = np.random.default_rng(seed)

    # ---- Pick a row from the master manifest ----
    if sample_id is not None:
        subset = master[master["sample_id"] == sample_id]
        if len(subset) == 0:
            raise ValueError(f"sample_id '{sample_id}' not found in master manifest.")
        row = subset.iloc[0]
    else:
        subset = master if cls is None else master[master["target_class"] == cls]
        if len(subset) == 0:
            raise ValueError(f"No samples for cls='{cls}'.")
        row = subset.iloc[int(rng.integers(0, len(subset)))]

    cls_name = row["target_class"]
    fold = int(row["outer_fold"])

    # ---- Load model + fold-specific calibration ----
    model = _load_fold_model(fold)
    fr = fold_results[fold_results["fold"].astype(int) == fold].iloc[0]
    bin_threshold = float(fr["bin_threshold"])
    prior_lambda  = float(fr["prior_lambda"])

    # ---- Build a one-row dataset item (no stain aug, no training aug) ----
    one_df = pd.DataFrame([row])
    ds = EBHIWeakOrdinalDataset(one_df, train=False, paired_stain=False)
    sample = ds[0]
    x = sample["clean"].unsqueeze(0).to(DEVICE)

    # ---- Inference with TTA + fold-calibrated (bin_thr, prior_lambda) ----
    with torch.no_grad():
        bin_logits, grade_logits = infer_logits(model, x, tta=True)
        sem_pred, bin_prob = semantic_from_logits(
            bin_logits.cpu(), grade_logits.cpu(),
            out_hw=(IMAGE_SIZE, IMAGE_SIZE),
            bin_threshold=bin_threshold,
            prior_lambda=prior_lambda,
        )
    sem_pred = sem_pred[0].numpy().astype(np.int32)  # HxW in {0..4}
    bin_prob = bin_prob[0].numpy()

    # ---- Reference weak semantic GT (foreground × slide-level grade) ----
    rgb = np.asarray(
        Image.open(row["image"]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
    )
    fg_mask = sample["mask"][0].numpy().astype(np.uint8)      # 0/1
    grade_idx1 = int(row["grade_idx"]) + 1                     # 1..4 for semantic map
    weak_gt = fg_mask * grade_idx1                             # HxW in {0, grade_idx1}

    # ---- Metrics on this one image ----
    def _dice(a, b):
        a = a.astype(bool); b = b.astype(bool)
        d = a.sum() + b.sum()
        return 1.0 if d == 0 else 2 * np.logical_and(a, b).sum() / d

    fg_pred = sem_pred > 0
    binary_dice   = _dice(fg_pred, fg_mask.astype(bool))
    grade_pred_mask = sem_pred == grade_idx1
    grade_gt_mask   = weak_gt == grade_idx1
    grade_dice    = _dice(grade_pred_mask, grade_gt_mask)

    total_fg = max(fg_pred.sum(), 1)
    composition = {
        SEMANTIC_LABELS[k]: float((sem_pred == k).sum()) / total_fg
        for k in range(1, 5)
    }

    result = {
        "sample_id":     row["sample_id"],
        "true_class":    GRADE_PRETTY[cls_name],
        "outer_fold":    fold,
        "bin_threshold": bin_threshold,
        "prior_lambda":  prior_lambda,
        "binary_dice":   binary_dice,
        "grade_dice":    grade_dice,
        "composition":   composition,
        "rgb":           rgb,
        "weak_gt":       weak_gt,
        "sem_pred":      sem_pred,
        "fg_mask":       fg_mask,
        "bin_prob":      bin_prob,
    }

    if not show:
        return result

    # ---- Plot: ORIGINAL | WEAK GT | MASOR PREDICTION | OVERLAY ----
    fig, ax = plt.subplots(1, 4, figsize=(20, 5.5))

    ax[0].imshow(rgb)
    ax[0].set_title(f"{result['true_class']}\n{row['sample_id']}\nfold {fold}", fontsize=11)

    ax[1].imshow(weak_gt, cmap=CMAP, vmin=0, vmax=4, interpolation="nearest")
    ax[1].set_title("Weak semantic reference", fontsize=11)

    ax[2].imshow(sem_pred, cmap=CMAP, vmin=0, vmax=4, interpolation="nearest")
    ax[2].set_title(
        f"MASOR-2.1 prediction\nFG Dice={binary_dice:.3f} | Grade Dice={grade_dice:.3f}",
        fontsize=11,
    )

    ax[3].imshow(rgb)
    ax[3].imshow(np.ma.masked_where(sem_pred == 0, sem_pred),
                 cmap=CMAP, vmin=0, vmax=4, alpha=0.55, interpolation="nearest")
    comp_str = " | ".join(f"{k} {v*100:.1f}%" for k, v in composition.items())
    ax[3].set_title(f"Prediction overlay\n{comp_str}", fontsize=10)

    for a in ax:
        a.set_xticks([]); a.set_yticks([])

    legend = [Patch(facecolor=SEMANTIC_COLORS[i], label=SEMANTIC_LABELS[i]) for i in range(5)]
    fig.legend(handles=legend, loc="lower center", ncol=5, fontsize=11, frameon=True)

    fig.suptitle(
        f"MASOR-2.1 — random held-out image ({result['true_class']}, fold {fold})",
        fontsize=14, fontweight="bold", y=1.02,
    )
    plt.tight_layout(rect=[0, 0.04, 1, 1.0])
    plt.show()

    print(
        f"[{row['sample_id']}] fold {fold}  "
        f"bin_thr={bin_threshold:.2f}  prior_lambda={prior_lambda:.2f}  "
        f"FG Dice={binary_dice:.3f}  Grade Dice={grade_dice:.3f}"
    )

    return result


# ------------------------------------------------------------
# Fire once so the cell renders something on run
# ------------------------------------------------------------
_ = run_one_random()

In [ ]:
from pathlib import Path
import os

print("=== What's in /kaggle/working ===")
for p in sorted(Path("/kaggle/working").rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f"  {p}  ({size_mb:.1f} MB)")

print("\n=== Looking for any .pt files anywhere ===")
for root in ["/kaggle/working", "/kaggle/input", "/kaggle/temp"]:
    r = Path(root)
    if r.exists():
        for p in r.rglob("*.pt"):
            print(f"  {p}")

print("\n=== Config expected path ===")
try:
    print(f"  WORK_ROOT       = {WORK_ROOT}")
    print(f"  CHECKPOINT_ROOT = {CHECKPOINT_ROOT}")
    print(f"  Exists: {CHECKPOINT_ROOT.exists()}")
except NameError:
    print("  WORK_ROOT not defined — rerun cell 3 (Configuration)")